Cell 1: Setup & Configuration
This cell initializes the project, checks for available hardware, and sets up a global configuration dictionary for all key parameters.

In [1]:
from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from mpl_toolkits.axes_grid1 import make_axes_locatable
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from tqdm.auto import tqdm
from transformers import SegformerForSemanticSegmentation
import albumentations as A
import cv2
import itertools
import math
import matplotlib.colors as mcolors
from matplotlib.colors import PowerNorm
import matplotlib.pyplot as plt
import numpy as np
import os
import re
import timm
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*resume_download is deprecated.*")
print("🔇 Annoying FutureWarnings successfully suppressed!")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE}")

CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,

    # Heads
    'num_seg_classes': 11,
    'num_det_classes': 91,

    # Stereo Geometry
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 48,

    # Training
    # ✅ CHANGE 1a: batch_size 8→4 (VRAM safety, Phase-2 hat Context-Net extra params)
    'batch_size': 4,
    'ACCUMULATION_STEPS': 12,
    'lr': 2e-4,
    # ✅ CHANGE 1b: 12 Epochen (6× Phase-2a + 6× Phase-2b)
    'num_epochs': 12,
    # ✅ CHANGE 1c: Ab dieser Epoche wird Seg aufgetaut + Dilated Conv aktiviert
    'PHASE2B_EPOCH': 6,
    # ✅ CHANGE 1d: num_workers 8→4 (stabiler mit weniger Prefetch-Puffer)
    'num_workers': 4,
    # ✅ CHANGE 1e: separates Checkpoint-Verzeichnis für Phase 2
    'save_dir': "./checkpoints_phase2"
}
os.makedirs(CONFIG['save_dir'], exist_ok=True)
print("✅ Setup complete. Phase-2 configuration loaded.")


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔇 Annoying FutureWarnings successfully suppressed!
🚀 Running on: cuda
✅ Setup complete. Phase-2 configuration loaded.


Cell 2: Fused Model Architecture
This cell contains the complete, final architecture for the FusedHexapodModel, including the shared MobileNetV3 backbone and all three specialized heads (Stereo, Segmentation, and Detection).

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math

# --- Constants for Fixed Resolutions (Hailo Requirement) ---
RES_S8 = (60, 80)
RES_S4 = (120, 160)
RES_S1 = (480, 640)

# --- Helper Blocks ---
class ConvBnReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, dilation=1):
        super().__init__()
        padding = ((kernel_size - 1) * dilation) // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBnReLU(channels, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.relu  = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.conv1(x))))

class ConvBR(nn.Module):
    def __init__(self, c_in, c_out, kernel=3, stride=1):
        super().__init__()
        padding = (kernel - 1) // 2
        self.conv = nn.Conv2d(c_in, c_out, kernel_size=kernel, stride=stride, padding=padding, bias=False)
        self.bn   = nn.BatchNorm2d(c_out)
        self.act  = nn.ReLU(inplace=True)
    def forward(self, x): return self.act(self.bn(self.conv(x)))

# --- Stereo Components ---
class CoarseCostVolume(nn.Module):
    def __init__(self, max_disp, in_channels):
        super().__init__()
        self.max_disp = max_disp
        self.compress = nn.Conv2d(in_channels, 16, 1, bias=False)
        self.reduce   = nn.Conv2d(16, 1, 1, bias=False)
        with torch.no_grad():
            self.reduce.weight.fill_(1.0 / 16)
            self.reduce.weight.requires_grad = False

    def forward(self, left_feat, right_feat):
        l = self.compress(left_feat)
        r = self.compress(right_feat)
        cost_list = []
        for d in range(self.max_disp):
            if d == 0:
                cost_list.append(self.reduce(l * r))
            else:
                sim = self.reduce(l[:, :, :, d:] * r[:, :, :, :-d])
                cost_list.append(F.pad(sim, (d, 0, 0, 0)))
        return torch.cat(cost_list, dim=1)

class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor):
        super().__init__()
        self.scale_factor = scale_factor
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),                    nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )
    def forward(self, disparity_low, guidance):
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor,
            mode='bilinear', align_corners=False
        ) * self.scale_factor
        return F.relu(disparity_up + self.net(torch.cat([disparity_up, guidance], dim=1)))

# ✅ CHANGE 2a: Context Network im HierarchicalStereoHead
#
# Problem in Phase 1: horizontale Streifenartefakte auf texturarmen Flächen
# Ursache: soft-argmax-Verteilung zu breit → ambigue Disparitäten nebeneinander
#           werden nicht räumlich koordiniert
#
# Lösung: Context Network schärft den Cost Volume VOR dem softmax
#   • groups=max_disp_s8 → Depthwise-Conv (jede Disp-Ebene separat, MobileNet-Style)
#   • 1×1 Conv nach Depthwise → Pointwise (Interaktion zwischen Ebenen)
#   • Identity-Init beim Laden aus Phase-1-Checkpoint (kein Loss-Spike)
#
# Hailo-8 kompatibel: Depthwise + Pointwise = Standard MobileNet-Block ✅
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.reduce_s8   = nn.Conv2d(ch_s8, 32, 1, bias=False)
        self.reduce_s4   = nn.Conv2d(ch_s4, 32, 1, bias=False)
        self.stereo_coarse    = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=32)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=32, scale_factor=2.0)
        self.stereo_refine_s1 = RefinementStage(guidance_channels=1,  scale_factor=4.0)
        self.register_buffer(
            'disp_reg',
            torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1)
        )

        # ✅ NEU: Context Network (Depthwise + Pointwise)
        self.context = nn.Sequential(
            nn.Conv2d(max_disp_s8, max_disp_s8, 3, padding=1, groups=max_disp_s8, bias=False),
            nn.Conv2d(max_disp_s8, max_disp_s8, 1, bias=True),
            nn.LeakyReLU(negative_slope=0.01, inplace=True)
        )

    def forward(self, l_s8, r_s8, l_s4, l_img_raw):
        feat_l_s8 = self.reduce_s8(l_s8)
        feat_r_s8 = self.reduce_s8(r_s8)
        feat_l_s4 = self.reduce_s4(l_s4)

        vol_s8  = self.stereo_coarse(feat_l_s8, feat_r_s8)
        vol_s8  = self.context(vol_s8)
        # ✅ Change A: Temperatur T=0.5 → spitzere Softmax-Verteilung → schärfere Disparitäten
        prob_s8 = F.softmax(vol_s8 / 0.7, dim=1)   # war 0.5
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)

        disp_s4    = self.stereo_refine_s4(disp_s8, feat_l_s4)
        final_disp = self.stereo_refine_s1(disp_s4, l_img_raw)
        # ✅ Change B: disp_s8 zurückgeben für Aux-Loss
        return final_disp, disp_s8
# ✅ CHANGE 2b: Dilated Conv in LRASPPHead für schärfere Seg-Details (Phase-2b)
#
# Problem: LRASPPHead sieht nur stride-4 (low) und stride-16 (high) Features
#          → mittlere Skala (stride-8) wird nicht genutzt → unscharfe Konturen
#
# Lösung: mid_classifier mit dilation=2 → größerer Receptive Field ohne ASPP-Overhead
#   • wird erst in Phase-2b aktiviert (use_mid = True)
#   • Phase-2a: Seg bleibt eingefroren → mid_classifier ungenutzt
#
# Hailo-8 kompatibel: einzelne dilated Conv2d wird unterstützt ✅
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AvgPool2d(kernel_size=(30, 40)),  # 480/16=30, 640/16=40
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.Sigmoid()
        )
        self.low_classifier  = nn.Conv2d(low_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)

        # ✅ NEU: Dilated Conv für mittlere Skala (Phase-2b)
        # Kernel 3×3, dilation 2 → effektiver Receptive Field 5×5
        self.mid_classifier = nn.Conv2d(128, num_classes, 3, padding=2, dilation=2)
        # Startet deaktiviert – wird in Phase-2b via model.seg_head.use_mid = True eingeschaltet
        self.use_mid = False

    def forward(self, x_low, x_high):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, scale_factor=4.0, mode='bilinear', align_corners=False)
        result = self.low_classifier(x_low) + self.high_classifier(out)
        # ✅ In Phase-2b: mittlere Disp-Ebene addieren
        if self.use_mid:
            result = result + self.mid_classifier(out)
        return result

class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, width=256):
        super().__init__()
        self.stem     = ConvBR(ch_in, width, kernel=1, stride=1)
        self.reg_conv = nn.Sequential(ConvBR(width, width), ConvBR(width, width))
        self.reg_pred = nn.Conv2d(width, 4, kernel_size=1)
        self.cls_conv = nn.Sequential(ConvBR(width, width), ConvBR(width, width))
        self.cls_pred = nn.Conv2d(width, num_classes, kernel_size=1)
        self.obj_pred = nn.Conv2d(width, 1, kernel_size=1)

    def forward(self, x):
        x     = self.stem(x)
        x_reg = self.reg_conv(x)
        x_cls = self.cls_conv(x)
        return torch.cat([self.reg_pred(x_reg), self.obj_pred(x_reg), self.cls_pred(x_cls)], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, ch_dims, num_classes):
        super().__init__()
        self.head_s8  = DecoupledHead(ch_dims[1], num_classes)
        self.head_s16 = DecoupledHead(ch_dims[2], num_classes)
        self.head_s32 = DecoupledHead(ch_dims[3], num_classes)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model(
            'mobilenetv3_large_100', pretrained=True,
            features_only=True, out_indices=(1, 2, 3, 4)
        )
        bb_ch = self.backbone.feature_info.channels()
        self.max_disp_s8 = 24
        self.stereo_head = HierarchicalStereoHead(
            ch_s8=bb_ch[1], ch_s4=bb_ch[0], max_disp_s8=self.max_disp_s8
        )
        self.seg_head  = LRASPPHead(low_ch=bb_ch[0], high_ch=bb_ch[2],
                                     num_classes=config['num_seg_classes'])
        self.yolo_head = YOLOHead(ch_dims=bb_ch, num_classes=config['num_det_classes'])

    def forward(self, left, right=None):
        x_left = left.repeat(1, 3, 1, 1) if left.shape[1] == 1 else left
        fl = self.backbone(x_left)
        final_disp = None
        if right is not None:
            x_right = right.repeat(1, 3, 1, 1) if right.shape[1] == 1 else right
            fr = self.backbone(x_right)
            l_img_gray = left.mean(dim=1, keepdim=True)
            # ✅ Tuple entpacken — disp_s8 wird hier verworfen (nur für Visualizer-Kompatibilität)
            final_disp, _ = self.stereo_head(
                l_s8=fl[1], r_s8=fr[1], l_s4=fl[0], l_img_raw=l_img_gray
            )
        seg_preds = self.seg_head(fl[0], fl[2])
        det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        return final_disp, seg_preds, det_preds

# ✅ CHANGE 2c: initialize_yolo_head() bleibt als Funktion definiert,
#               wird aber NICHT mehr aufgerufen – Phase 2 lädt alles aus Checkpoint!
def initialize_yolo_head(model, prior_prob=0.002):
    bias_value = -math.log((1 - prior_prob) / prior_prob)
    count = 0
    for m in model.modules():
        if isinstance(m, DecoupledHead):
            for layer in m.reg_conv:
                if isinstance(layer, ConvBR):
                    nn.init.kaiming_normal_(layer.conv.weight, mode='fan_out', nonlinearity='relu')
            nn.init.normal_(m.reg_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.reg_pred.bias, 0.0)
            for layer in m.cls_conv:
                if isinstance(layer, ConvBR):
                    nn.init.kaiming_normal_(layer.conv.weight, mode='fan_out', nonlinearity='relu')
            nn.init.normal_(m.cls_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.cls_pred.bias, bias_value)
            nn.init.normal_(m.obj_pred.weight, mean=0.0, std=0.01)
            nn.init.constant_(m.obj_pred.bias, bias_value)
            count += 1
    print(f"✅ Initialized {count} YOLO Heads with Prior Bias ({prior_prob}).")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FusedHexapodModel(CONFIG).to(DEVICE)
# ✅ KEIN initialize_yolo_head() Aufruf hier — Gewichte kommen aus Phase-1-Checkpoint!

for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.momentum = 0.01

print("✅ Phase-2 FusedHexapodModel ready (Context Network + Dilated Seg).")


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ Phase-2 FusedHexapodModel ready (Context Network + Dilated Seg).


Cell 3: Unified Loss Function
This cell contains the corrected and unified loss function. It properly calculates the stereo loss on the refined output and combines it with segmentation and detection losses.

In [3]:
# --- 1. Define Helper Blocks ---
class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes, stride):
        super().__init__()
        self.num_classes = num_classes
        self.stride = stride
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.l1 = nn.L1Loss(reduction='none')

    '''
    def _get_targets(self, targets, cls_preds, reg_preds):
        """
        Erzeugt die Target-Tensoren für Objectness, Regression und Klassifizierung.
        
        Encoding:
        - Offsets (dx, dy): Subpixel-Lage relativ zur Grid-Zelle
        - Größen (lw, lh): Logarithmierte Breite/Höhe in Grid-Einheiten
        """
        import math # Just in case it's not imported at the top of your file
        
        B, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        # Initialisierung der Target-Tensoren
        cls_t = torch.zeros_like(cls_preds)
        reg_t = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((B, 1, H, W), device=device) 

        for b in range(B):
            if targets[b].numel() == 0: 
                continue
            
            # targets[b] Format: [N, 5] -> [class, xc, yc, w, h] (normalisiert 0-1)
            gt_boxes = targets[b].clone()
            
            # Koordinaten von Normalisiert [0, 1] auf Grid-Skala [0, W/H] umrechnen
            gt_boxes[:, 1] *= W  # xc in Grid-Einheiten
            gt_boxes[:, 2] *= H  # yc in Grid-Einheiten
            gt_boxes[:, 3] *= W  # w  in Grid-Einheiten
            gt_boxes[:, 4] *= H  # h  in Grid-Einheiten
            
            for box in gt_boxes:
                cls_id, gx, gy, gw, gh = box.tolist()
                
                # Grid-Zelle bestimmen (Integer-Anteil des wahren Zentrums)
                ix, iy = int(gx), int(gy)
                
                # Vorbereiten der logarithmischen Größen (für alle Nachbarn gleich)
                lw = math.log(max(gw, 1e-6))
                lh = math.log(max(gh, 1e-6))
                cid = int(cls_id)
                
                # 🚨 3x3 Cross Assignment (Zentrum + Oben, Unten, Links, Rechts)
                for dx, dy in [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = ix + dx, iy + dy
                    
                    # Boundary Check: Liegt der Nachbar innerhalb des Grids?
                    if 0 <= nx < W and 0 <= ny < H:
                        
                        # 1. Objectness Target setzen
                        obj_mask[b, 0, ny, nx] = 1.0
                        
                        # 2. Regression Targets (dx, dy MÜSSEN relativ zum Nachbarn nx, ny sein!)
                        reg_t[b, 0, ny, nx] = gx - nx
                        reg_t[b, 1, ny, nx] = gy - ny
                        reg_t[b, 2, ny, nx] = lw
                        reg_t[b, 3, ny, nx] = lh
                        
                        # 3. Classification Target
                        if 0 <= cid < self.num_classes:
                            cls_t[b, cid, ny, nx] = 1.0

        return cls_t, reg_t, obj_mask
    '''
    def _get_targets(self, targets, cls_preds, reg_preds):
        """
        Vektorisierte Version:
        Erzeugt die Target-Tensoren für Objectness, Regression und Klassifizierung
        komplett ohne For-Schleifen über Boxen oder Nachbarn.
        """
        B, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        # Initialisierung der Target-Tensoren
        cls_t = torch.zeros_like(cls_preds)
        reg_t = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((B, 1, H, W), device=device) 
        
        # 1. Targets flachklopfen und Batch-Index hinzufügen
        batch_targets = []
        for b in range(B):
            if targets[b].numel() > 0:
                t = targets[b].clone()
                # Batch-Index als erste Spalte hinzufügen: [b, class_id, xc, yc, w, h]
                b_idx = torch.full((t.shape[0], 1), b, device=device, dtype=t.dtype)
                batch_targets.append(torch.cat([b_idx, t], dim=1))
                
        # Wenn der ganze Batch leer ist, sind wir fertig
        if not batch_targets:
            return cls_t, reg_t, obj_mask
            
        gt = torch.cat(batch_targets, dim=0) # Shape: [Alle_Boxen_im_Batch, 6]
        
        # Variablen extrahieren (alles auf einmal!)
        b_idx  = gt[:, 0].long()
        cls_id = gt[:, 1].long()
        gx     = gt[:, 2] * W
        gy     = gt[:, 3] * H
        gw     = gt[:, 4] * W
        gh     = gt[:, 5] * H
        
        # Basis-Grid-Zelle
        ix = gx.long()
        iy = gy.long()
        
        # 2. 5 Nachbarn generieren (Cross Assignment)
        # Offsets für Zentrum, Links, Rechts, Oben, Unten
        offsets = torch.tensor([
            [0, 0], [-1, 0], [1, 0], [0, -1], [0, 1]
        ], device=device, dtype=torch.long)
        
        # Wir berechnen die Nachbar-Koordinaten (nx, ny) für alle Boxen gleichzeitig
        # unsqueeze() hilft uns, die Matrix zu erweitern und offsets zu addieren
        nx = (ix.unsqueeze(1) + offsets[:, 0].unsqueeze(0)).flatten()
        ny = (iy.unsqueeze(1) + offsets[:, 1].unsqueeze(0)).flatten()
        
        # Da wir nun aus jeder Box 5 Nachbarn gemacht haben, 
        # müssen wir die anderen Werte (Batch-ID, Zielwerte) ebenfalls 5-mal wiederholen.
        b_idx_rep  = b_idx.repeat_interleave(5)
        cls_id_rep = cls_id.repeat_interleave(5)
        gx_rep     = gx.repeat_interleave(5)
        gy_rep     = gy.repeat_interleave(5)
        gw_rep     = gw.repeat_interleave(5)
        gh_rep     = gh.repeat_interleave(5)
        
        # 3. Boundary Check AND Class Check
        num_classes = cls_preds.shape[1]  # Hole die Anzahl der Klassen vom Tensor
        
        valid_mask = (
            (nx >= 0) & (nx < W) & 
            (ny >= 0) & (ny < H) & 
            (cls_id_rep >= 0) & (cls_id_rep < num_classes) # 🚨 NEU: Verhindert CUDA Crash!
        )
        
        # Wir behalten nur die Werte, wo valid_mask True ist
        b_v   = b_idx_rep[valid_mask]
        c_v   = cls_id_rep[valid_mask]
        nx_v  = nx[valid_mask]
        ny_v  = ny[valid_mask]
        gx_v  = gx_rep[valid_mask]
        gy_v  = gy_rep[valid_mask]
        gw_v  = gw_rep[valid_mask]
        gh_v  = gh_rep[valid_mask]
        
        # 4. Vorbereiten der logarithmischen Größen
        lw_v = torch.log(torch.clamp(gw_v, min=1e-6))
        lh_v = torch.log(torch.clamp(gh_v, min=1e-6))
        
        # 5. Werte in die Tensoren schreiben
        obj_mask[b_v, 0, ny_v, nx_v] = 1.0
        
        reg_t[b_v, 0, ny_v, nx_v] = gx_v - nx_v
        reg_t[b_v, 1, ny_v, nx_v] = gy_v - ny_v
        reg_t[b_v, 2, ny_v, nx_v] = lw_v
        reg_t[b_v, 3, ny_v, nx_v] = lh_v
        
        cls_t[b_v, c_v, ny_v, nx_v] = 1.0
        
        return cls_t, reg_t, obj_mask
        
    def sigmoid_focal_loss(self, inputs, targets, alpha=0.75, gamma=2.0, reduction='none'): # ⬅️ Changed default to 0.75
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce_loss * ((1 - p_t) ** gamma)

        if alpha >= 0:
            # If target=1, alpha_t = 0.75. If target=0, alpha_t = 0.25. 
            # This properly boosts the rare positive anchors!
            alpha_t = alpha * targets + (1 - alpha) * (1 - targets) 
            loss = alpha_t * loss

        if reduction == "mean":
            return loss.mean()
        elif reduction == "sum":
            return loss.sum()
        else:
            return loss

    def forward(self, preds, targets):
        # Slice components: [B, 5 + Num_Classes, H, W]
        reg_p = preds[:, :4, :, :]   # [dx, dy, log_w, log_h]
        obj_p = preds[:, 4:5, :, :]  # [objectness]
        cls_p = preds[:, 5:, :, :]   # [classes]

        # 1. Get Ground Truth Targets
        cls_t, reg_t, obj_mask = self._get_targets(targets, cls_p, reg_p)
        
        # 🚨 METRICS EXTRACTION FOR TENSORBOARD
        num_targets_val = obj_mask.sum().item()
        max_cls_val = cls_t.max().item() if num_targets_val > 0 else 0

        num_pos = torch.clamp(obj_mask.sum(), min=1.0)

        # 🚨 Protect against empty TartanAir targets overwriting the weights
        if num_targets_val == 0:
            return {
                'total': torch.tensor(0.0, requires_grad=True, device=preds.device),
                'box': 0.0, 'obj': 0.0, 'cls': 0.0,
                'num_targets': 0,     # Added
                'max_cls': 0          # Added
            }
        
        # 2. OBJECTNESS LOSS (mit Focal Loss für bessere Balance)
        l_obj = self.sigmoid_focal_loss(obj_p, obj_mask, reduction='sum') / num_pos

        # 3. REGRESSION LOSS (nur auf positiven Samples)
        # WICHTIG: Separate Gewichtung für Offsets vs. Größen
        # VORAUSSETZUNG: self.l1 = nn.L1Loss(reduction='none')
        l_offset = (self.l1(reg_p[:, :2], reg_t[:, :2]) * obj_mask).sum() / num_pos
        l_size = (self.l1(reg_p[:, 2:], reg_t[:, 2:]) * obj_mask).sum() / num_pos
        l_box = l_offset + 2.0 * l_size  # Größen sind wichtiger

        # 4. CLASSIFICATION LOSS (BUG FIXED: Nur auf positiven Samples!)
        l_cls_raw = self.sigmoid_focal_loss(cls_p, cls_t, reduction='none')
        l_cls = (l_cls_raw * obj_mask).sum() / num_pos

        # ANGEPASSTE Gewichtung: Box-Loss ist jetzt wichtiger
        return {
            'total': 5.0 * l_box + 5.0 * l_obj + l_cls, 
            'box': l_box.item(),
            'obj': l_obj.item(),
            'cls': l_cls.item(),
            'num_targets': num_targets_val,  # ⬅️ To TensorBoard
            'max_cls': max_cls_val           # ⬅️ To TensorBoard
        }

class EdgeAwareSmoothnessLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, disp, img):
        mean_disp = disp.mean(2, True).mean(3, True)
        disp = disp / (mean_disp + 1e-7)
        
        grad_disp_x = torch.abs(disp[:, :, :, :-1] - disp[:, :, :, 1:])
        grad_disp_y = torch.abs(disp[:, :, :-1, :] - disp[:, :, 1:, :])

        grad_img_x = torch.mean(torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:]), 1, keepdim=True)
        grad_img_y = torch.mean(torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]), 1, keepdim=True)

        grad_img_x = torch.exp(-torch.mean(grad_img_x, 1, keepdim=True))
        grad_img_y = torch.exp(-torch.mean(grad_img_y, 1, keepdim=True))

        return torch.mean(grad_disp_x * grad_img_x) + torch.mean(grad_disp_y * grad_img_y)


# --- 2. Main Loss Class ---
class FusedHexapodLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        # YOLO Skalen-Verluste
        self.yolo_s8_loss  = SimpleYOLOLoss(config['num_det_classes'], stride=8)
        self.yolo_s16_loss = SimpleYOLOLoss(config['num_det_classes'], stride=16)
        self.yolo_s32_loss = SimpleYOLOLoss(config['num_det_classes'], stride=32)
        
        # KEIN seg_loss (CrossEntropy gegen GT) mehr hier!
        self.kl_div = nn.KLDivLoss(reduction='batchmean') 
        self.smooth_loss = EdgeAwareSmoothnessLoss()
        
        # Basis-Gewichte (werden durch UncertaintyWeighting modifiziert)
        self.w_yolo = 1.0 
        self.w_kd = 1.0     # Fokus liegt jetzt auf KD für Segmentation
        self.w_stereo = 1.0

    def forward(self, preds, targets, teacher_preds, left_img=None):
        # Entpacken der Vorhersagen
        stereo_preds, seg_preds, det_preds = preds
        
        task_tensors = {} # Für Gradienten (Tensors)
        logs = {}         # Für Tensorboard (Floats)

        # --- 1. STEREO LOSS ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                # ✅ CHANGE 3a: Dynamisches Smooth-L1-Beta
                # Problem: beta=1.0 (default) → linearer Gradient für alle Fehler >1px
                #          Früh im Training (L1≈18): zu aggressiv → Instabilität
                #          Spät im Training (L1≈1.5): zu klein → stumpfer Gradient
                # Lösung: beta = max(1.0, L1_loss * 0.5)
                #   Early  (L1≈18) → beta≈9   → quadratischer Bereich bis 9px
                #   Late   (L1≈1.5)→ beta=1.0 → Standard
                with torch.no_grad():
                    current_beta = max(1.0, F.l1_loss(stereo_preds[mask], gt_disp[mask]).item() * 0.5)
                l_stereo_raw = F.smooth_l1_loss(stereo_preds[mask], gt_disp[mask], beta=current_beta)
                
                # 2. Add Edge-Aware Smoothness
                if left_img is not None:
                    l_smooth = self.smooth_loss(stereo_preds, left_img)
                    
                    # FIX: Instead of a hard 0.02, scale it relative to the raw L1 loss.
                    # This prevents the smoothness penalty from dominating when the network
                    # gets close to convergence, allowing sharp details to emerge.
                    dynamic_smooth_weight = torch.clamp(l_stereo_raw.detach() * 0.01, max=0.02)
                    l_stereo = l_stereo_raw + dynamic_smooth_weight * l_smooth
                    
                    logs['smooth'] = l_smooth.item()
                else:
                    l_stereo = l_stereo_raw
                
                # ✅ CHANGE 3b: Loss-Vorskalierung Stereo (/2)
                # Skaliert Gradient VOR dem Clipping → Backbone bekommt proportionalen Anteil
                # (LR-Reduktion würde erst NACH Clipping greifen → Clipping feuert weiterhin)
                task_tensors['stereo'] = l_stereo / 2.0
                logs['stereo'] = l_stereo.item()

        # --- 2. KNOWLEDGE DISTILLATION (KD) ---
        if teacher_preds is not None and seg_preds is not None:
            T = 2.0 
            
            # Student Logits auf Teacher-Größe
            s_logits = F.interpolate(seg_preds, size=teacher_preds.shape[-2:], 
                                     mode='bilinear', align_corners=False)
            
            # STUDENT: Log-Softmax (Korrekt für KLDiv)
            p_s = F.log_softmax(s_logits / T, dim=1)
            
            # TEACHER: Numerisch stabilere Berechnung
            # Wir nehmen an, teacher_preds sind LOGITS (rohe Scores).
            # Statt exp() -> log() -> softmax() machen wir direkt Softmax auf den Logits.
            p_t = F.softmax(teacher_preds / T, dim=1)
            
            # KL Divergenz BERECHNUNG
            # WICHTIG: reduction='none', damit wir selbst über Pixel mitteln können
            kl_loss_pixelwise = F.kl_div(p_s, p_t, reduction='none') * (T**2)
            
            # Jetzt die Magie: 
            # 1. Summe über Klassen (dim=1) -> Das ist die KL-Div pro Pixel
            # 2. Mittelwert über Spatial (dim=2,3) & Batch (dim=0) -> Skalenunabhängig!
            l_kd = kl_loss_pixelwise.sum(dim=1).mean()
            
            task_tensors['seg'] = l_kd 
            logs['seg_kd'] = l_kd.item()
            
        # --- 3. YOLO LOSS ---
        if det_preds is not None and targets.get('det') is not None:
            gt_det = targets['det']
            
            # Alle 3 Skalen berechnen
            r8  = self.yolo_s8_loss(det_preds[0], gt_det)
            r16 = self.yolo_s16_loss(det_preds[1], gt_det)
            r32 = self.yolo_s32_loss(det_preds[2], gt_det)
            
            l_yolo = (r8['total'] + r16['total'] + r32['total']) / 3.0
            
            # ✅ CHANGE 3b: Loss-Vorskalierung YOLO (/10)
            # Ohne Skalierung: YOLO-Loss ≈17 vs Stereo ≈3.4 → UW treibt log_var_yolo auf 3.5
            # Mit /10: YOLO ≈1.7, ähnlich Stereo → UW-Gleichgewicht bei log_var ≈ 0.0–0.7
            task_tensors['yolo'] = l_yolo / 10.0
            logs['yolo'] = l_yolo.item()
            
            # Helper to safely extract floats whether it's a tensor or a primitive float
            def _to_float(val): return val.item() if isinstance(val, torch.Tensor) else float(val)
            
            logs['yolo_box'] = _to_float((r8['box'] + r16['box'] + r32['box']) / 3.0)
            logs['yolo_obj'] = _to_float((r8['obj'] + r16['obj'] + r32['obj']) / 3.0)
            logs['yolo_cls'] = _to_float((r8['cls'] + r16['cls'] + r32['cls']) / 3.0)

            # 🚨 FIX: Extract the debug metrics and pass them up!
            # Sum the targets across all 3 scales
            logs['yolo_num_targets'] = r8.get('num_targets', 0) + r16.get('num_targets', 0) + r32.get('num_targets', 0)
            
            # Get the highest class ID found across all 3 scales
            logs['yolo_max_cls'] = max(r8.get('max_cls', 0), r16.get('max_cls', 0), r32.get('max_cls', 0))

        return task_tensors, logs
print("✅ Loss function updated.")

✅ Loss function updated.


Cell 4: Data Parsing Helpers
This cell contains the necessary helper functions to scan the respective dataset directories (FlyingThings3D, TartanAir, and COCO) and create a unified list of file paths and labels. The RealFusedDataset class in the next cell will use these functions.

In [4]:
import os
import glob
import numpy as np
from pycocotools.coco import COCO

def parse_ft3d(root):
    """ 
    Scans the FlyingThings3D directory for stereo pairs and disparity maps. 
    Adapted for flattened structure: .../image_clean/left/*.png
    """
    print("   Scanning FlyingThings3D...")
    samples = []
    
    # Pfade basierend auf deiner Struktur
    img_dir_l = os.path.join(root, 'frames_cleanpass', 'TRAIN', 'image_clean', 'left')
    disp_dir_l = os.path.join(root, 'disparity', 'TRAIN', 'disparity', 'left')
    
    if not os.path.exists(img_dir_l):
        print(f"   [Error] Could not find directory: {img_dir_l}")
        return []
    
    # Glob alle PNGs
    left_files = sorted(glob.glob(os.path.join(img_dir_l, '*.png')))
    
    if not left_files:
        print(f"   [Warning] Directory found but no .png files inside: {img_dir_l}")
        return []

    # Matching mit Rechts & Disparity
    for l_path in left_files:
        filename = os.path.basename(l_path)
        
        # Rechtes Bild finden (String Replace 'left' -> 'right')
        # Vorsicht: Wir ersetzen nur das letzte Vorkommen oder nutzen os.sep
        parent_dir = os.path.dirname(l_path)
        if 'left' in parent_dir:
            r_path = l_path.replace(os.sep + 'left' + os.sep, os.sep + 'right' + os.sep)
        else:
            # Fallback, falls Pfadstruktur anders ist
            r_path = l_path.replace('left', 'right')
        
        # Disparity Pfad (.pfm)
        d_filename = filename.replace('.png', '.pfm')
        d_path = os.path.join(disp_dir_l, d_filename)
        
        if os.path.exists(r_path) and os.path.exists(d_path):
            samples.append({
                'type': 'stereo',
                'source': 'ft3d',
                'l': l_path,
                'r': r_path,
                'd': d_path,
                's': None, # Kein Seg
                'b': None  # Keine Boxen
            })
            
    print(f"   -> Found {len(samples)} FT3D pairs.")
    return samples

def parse_tartan(root):
    """ Scans the TartanAir directory for stereo pairs, depth, and optional segmentation. """
    print("   Scanning TartanAir...")
    samples = []
    
    # Suche rekursiv nach 'image_left' Ordnern
    # TartanAir Struktur: .../Environment/Easy/P000/image_left/...
    search_pattern = os.path.join(root, '**', 'image_left')
    left_folders = glob.glob(search_pattern, recursive=True)
    
    if not left_folders:
        print(f"   [Warning] No 'image_left' folders found in {root}")
        return []

    for l_folder in left_folders:
        # Parent ist z.B. .../P000/
        parent = os.path.dirname(l_folder)
        
        # Iteriere über Dateien im linken Ordner
        for f in os.listdir(l_folder):
            if not f.endswith('.png'): continue
            
            l_path = os.path.join(l_folder, f)
            
            # TartanAir Naming Convention: 
            # Left:  000000_left.png
            # Right: 000000_right.png
            r_filename = f.replace('_left', '_right')
            r_path = os.path.join(parent, 'image_right', r_filename)
            
            # Depth: 000000_left_depth.npy
            d_filename = f.replace('.png', '_depth.npy')
            d_path = os.path.join(parent, 'depth_left', d_filename)
            
            # Seg: 000000_left_seg.npy
            s_filename = f.replace('.png', '_seg.npy')
            s_path = os.path.join(parent, 'seg_left', s_filename)
            
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({
                    'type': 'stereo', 
                    'source': 'tartan',
                    'l': l_path, 
                    'r': r_path, 
                    'd': d_path, 
                    's': s_path if os.path.exists(s_path) else None, 
                    'b': None
                })
                
    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco(root):
    """ Scans the COCO directory for images and bounding box annotations. """
    print("   Scanning COCO 2017...")
    samples = []
    
    # Pfade prüfen
    ann_file = os.path.join(root, 'annotations', 'instances_train2017.json')
    img_dir = os.path.join(root, 'train2017')
    
    if not os.path.exists(ann_file): 
        print(f"   [Error] Annotation file not found: {ann_file}")
        return []
    
    try:
        coco = COCO(ann_file)
    except Exception as e:
        print(f"   [Error] Failed to load COCO JSON: {e}")
        return []

    # Kategorien filtern? (Optional, hier nehmen wir alle)
    # cat_ids = coco.getCatIds(catNms=['person', 'car', ...])
    # img_ids = coco.getImgIds(catIds=cat_ids)
    img_ids = coco.getImgIds()

    for iid in img_ids:
        img_info = coco.loadImgs(iid)[0]
        path = os.path.join(img_dir, img_info['file_name'])
        
        if not os.path.exists(path): continue
        
        # Annotations laden
        ann_ids = coco.getAnnIds(imgIds=iid, iscrowd=False)
        anns = coco.loadAnns(ann_ids)
        
        boxes = []
        H, W = img_info['height'], img_info['width']
        
        for ann in anns:
            # COCO bbox: [x_top_left, y_top_left, width, height]
            x, y, w, h = ann['bbox']
            
            # Validierung: Keine leeren Boxen
            if w < 1 or h < 1: continue

            # Convert to YOLO format [class, x_center, y_center, width, height] (Normalized)
            # category_id in COCO ist nicht kontinuierlich (1...90), wir müssen mappen falls nötig.
            # Hier: Wir nehmen an, num_classes deckt max(category_id) ab oder wir mappen später.
            # Einfaches Mapping: id - 1 (Vorsicht bei COCO, manche IDs fehlen!)
            cls_id = ann['category_id'] - 1 
            
            # YOLO Format: Center-based
            cx = (x + w / 2.0) / W
            cy = (y + h / 2.0) / H
            nw = w / W
            nh = h / H
            
            boxes.append([cls_id, cx, cy, nw, nh])
            
        if boxes:
            # Mono-Sample (nur Links, kein Rechts/Disp)
            samples.append({
                'type': 'mono',
                'source': 'coco',
                'l': path,
                'b': np.array(boxes, dtype=np.float32),
                'r': None,
                'd': None,
                's': None
            })
            
    print(f"   -> Found {len(samples)} COCO samples.")
    return samples

print("✅ Data parsing helper functions are ready.")


✅ Data parsing helper functions are ready.


Cell 5: Data Loading & Augmentation
This cell defines the RealFusedDataset class. It uses the parsers from the previous cell to build a master file list and then applies appropriate augmentations for training. It also includes the corrected .pfm file reader.

In [5]:
def read_pfm_fixed(file_path):
    """ Reads a .pfm file and returns a numpy array. Includes scaling. """
    with open(file_path, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        endian = '<' if scale < 0 else '>'
        scale = abs(scale)
        
        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data) 
    return (data * scale).copy()

import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

class RealFusedDataset(Dataset):
    def __init__(self, roots, img_size=(480, 640), mode='train'):
        """
        Pure CPU DataLoader for high-speed multiprocessing.
        GPU Teacher inference is completely removed from here and moved to the training loop.
        """
        self.roots = roots
        self.img_size = img_size  # (Height, Width) for PyTorch
        self.mode = mode
        
        self.samples = []
        
        # 1. SCAN DIRECTORIES (Assuming parse_tartan and parse_coco return dicts with 'source' key)
        if 'tartan' in self.roots:
            print("   Scanning TartanAir...")
            self.samples.extend(parse_tartan(roots['tartan']))
            
        if 'coco' in self.roots:
            print("   Scanning COCO...")
            self.samples.extend(parse_coco(roots['coco']))

        # ImageNet Stats for Normalization
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def _to_tensor(self, img_rgb):
        return torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0

    def __getitem__(self, idx):
        sample = self.samples[idx]
        source = sample.get('source') 
        
        # 🚨 FIX: OpenCV explicitly needs (Width, Height) for resizing!
        cv2_size = (self.img_size[1], self.img_size[0]) 
        
        # =========================================================
        # 1. TARTANAIR LOADING LOGIC
        # =========================================================
        if source == 'tartan':
            # --- 1. Load & Resize Images ---
            img_l_raw = cv2.imread(sample['l'])
            img_r_raw = cv2.imread(sample['r'])
            
            # Safety Check for corrupted files
            if img_l_raw is None or img_r_raw is None:
                print(f"❌ Corrupt Image: {sample['l']}")
                return self.__getitem__(0)
                
            img_l_rgb = cv2.cvtColor(img_l_raw, cv2.COLOR_BGR2RGB)
            img_r_rgb = cv2.cvtColor(img_r_raw, cv2.COLOR_BGR2RGB)

            l_img_aug = cv2.resize(img_l_rgb, cv2_size, interpolation=cv2.INTER_LINEAR)
            r_img_aug = cv2.resize(img_r_rgb, cv2_size, interpolation=cv2.INTER_LINEAR)
            
            t_l_rgb = self._to_tensor(l_img_aug)
            t_r_rgb = self._to_tensor(r_img_aug)
            
            # Teacher gets normalized RGB (Needs full color to segment)
            teacher_input = (t_l_rgb - self.mean) / self.std
            
            # Robot gets normalized Grayscale (Simulating IMX296 global shutter on Pi 5)
            gray_l = 0.299 * t_l_rgb[0] + 0.587 * t_l_rgb[1] + 0.114 * t_l_rgb[2]
            gray_r = 0.299 * t_r_rgb[0] + 0.587 * t_r_rgb[1] + 0.114 * t_r_rgb[2]
            robot_l = (gray_l.unsqueeze(0).repeat(3, 1, 1) - self.mean) / self.std
            robot_r = (gray_r.unsqueeze(0).repeat(3, 1, 1) - self.mean) / self.std

            # --- 2. Load Depth & Convert to Disparity ---
            if sample.get('d') and sample['d'].endswith('.pfm'):
                 raw_geo = read_pfm_fixed(sample['d']) 
            elif sample.get('d'):
                 raw_geo = np.load(sample['d'])
            else:
                 raw_geo = np.full((img_l_raw.shape[0], img_l_raw.shape[1]), -1.0, dtype=np.float32)

            # Convert Depth to Disparity (focal_length * baseline ≈ 80.0)
            valid_mask = raw_geo > 1e-4 
            disp = np.zeros_like(raw_geo)
            disp[valid_mask] = 80.0 / raw_geo[valid_mask] 
            disp = np.clip(disp, 0, 192)
            
            # Resize with NEAREST so we don't blur depth edges
            disp_aug = cv2.resize(disp, cv2_size, interpolation=cv2.INTER_NEAREST)
            disp_tensor = torch.from_numpy(disp_aug).unsqueeze(0).float() # [1, H, W]

            # --- 3. Load Segmentation ---
            seg = np.load(sample['s']) if sample.get('s') else np.full((img_l_raw.shape[0], img_l_raw.shape[1]), 255, dtype=np.uint8)
            seg_aug = cv2.resize(seg, cv2_size, interpolation=cv2.INTER_NEAREST)
            seg_tensor = torch.from_numpy(seg_aug).long() # [H, W]
            
            return {
                'left': robot_l,
                'right': robot_r,
                'teacher': teacher_input,
                'disp': disp_tensor,
                'seg': seg_tensor,
                'det': None,  # No YOLO GT in TartanAir, GPU Teacher will handle this!
                'use_stereo': 1.0,
                'use_seg': 1.0,
                'use_yolo': 0.0 # Will be overridden in training loop when pseudo-labels are generated
            }

        # =========================================================
        # 2. COCO LOADING LOGIC
        # =========================================================
        elif source == 'coco':
            # --- 1. Load & Resize Image ---
            img_l_raw = cv2.imread(sample['l'])
            if img_l_raw is None:
                print(f"❌ Corrupt Image: {sample['l']}")
                return self.__getitem__(0)
                
            img_l_rgb = cv2.cvtColor(img_l_raw, cv2.COLOR_BGR2RGB)
            l_img_aug = cv2.resize(img_l_rgb, cv2_size, interpolation=cv2.INTER_LINEAR)
            
            t_l_rgb = self._to_tensor(l_img_aug)
            
            # Normalizations
            teacher_input = (t_l_rgb - self.mean) / self.std
            gray_l = 0.299 * t_l_rgb[0] + 0.587 * t_l_rgb[1] + 0.114 * t_l_rgb[2]
            robot_l = (gray_l.unsqueeze(0).repeat(3, 1, 1) - self.mean) / self.std

            # --- 2. Load Ground Truth Bounding Boxes ---
            det_tensor = torch.zeros((0, 5)) # Fallback
            if sample.get('b') is not None:
                raw_boxes = np.array(sample['b'])
                if len(raw_boxes) > 0:
                    valid_b = (raw_boxes[:, 3] > 1e-4) & (raw_boxes[:, 4] > 1e-4)
                    clean_boxes = raw_boxes[valid_b]
                    if len(clean_boxes) > 0:
                        # Clip x, y, w, h to [0, 1] bounds just in case
                        clean_boxes[:, 1:] = np.clip(clean_boxes[:, 1:], 0.0, 1.0)
                        det_tensor = torch.tensor(clean_boxes, dtype=torch.float32)

            return {
                'left': robot_l,
                'right': robot_l.clone(),      # Dummy data to satisfy network dimensions
                'teacher': teacher_input,
                'disp': None,                  # COCO has no depth
                'seg': None,                   # COCO has no segmentation
                'det': det_tensor,             # Real GT boxes
                'use_stereo': 0.0,
                'use_seg': 0.0,
                'use_yolo': 1.0
            }

print("✅ RealFusedDataset class is ready.")

✅ RealFusedDataset class is ready.


Cell6: The Hexapod Visualizer
This function takes a batch and the model predictions, then displays the Left Image, Stereo Disparity, Semantic Map (Hexapod Physics), and YOLO Detections side-by-side.

In [6]:
import matplotlib.colors as mcolors

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch

def visualize_hexapod_output(step, writer):
    """
    Dein Original-Dashboard.
    Layout: 2 Zeilen, 4 Spalten.
    Zusatz: Saubere Scopes für YOLO-Metriken, um Scope-Leaks zu vermeiden.
    """
    was_training = model.training
    model.eval()
    
    VIS_THRESH = 0.15 
    
    # 🚨 CLEANUP: Initialize logging trackers explicitly BEFORE the loop
    yolo_final_conf_max = 0.0
    yolo_num_candidates = 0
    yolo_boxes_drawn = 0
    has_yolo_stats = False
    
    with torch.no_grad():
        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        
        for row, (name, b) in enumerate(static_batches.items()):
            l = b['left'].to(DEVICE)
            r = b['right'].to(DEVICE)
            t_img = b['teacher'].to(DEVICE)
            
            # Forward pass
            final_disp, seg_preds, det_preds = model(l, r)
            
            disp_pred = final_disp 
            disp_gt = b.get('disp') if b.get('disp') is not None else torch.zeros_like(l[:, 0, :, :])
            
            # --- COLUMN 1: GT / INPUT ---
            ax1 = axes[row, 0]
            img_bg = l[0].permute(1,2,0).cpu().numpy()
            img_bg = (img_bg - img_bg.min()) / (img_bg.max() - img_bg.min() + 1e-5)
            
            if name == 'yolo':
                ax1.imshow(img_bg)
                h, w = img_bg.shape[:2]
                if 'det' in b and b['det'] is not None and len(b['det']) > 0 and b['det'][0] is not None:
                    if len(b['det'][0]) > 0:
                        for box in b['det'][0]:
                            cls_id, xc, yc, bw, bh = box.tolist()
                            x1_box, y1_box = (xc - bw/2)*w, (yc - bh/2)*h  
                            box_color = plt.cm.tab10(int(cls_id) % 10)
                            rect = plt.Rectangle((x1_box, y1_box), bw*w, bh*h, fill=False, color=box_color, linewidth=2)
                            ax1.add_patch(rect)
                ax1.set_title("GT YOLO BOXES (Input)")
                vmax_scale = CONFIG['max_disp_pixel']
            else:
                disp_gt_np = disp_gt[0].squeeze().cpu().numpy()
                if disp_gt_np.max() > 0:
                    dynamic_vmax_gt = np.percentile(disp_gt_np[disp_gt_np > 0], 98)
                else:
                    dynamic_vmax_gt = 1.0 
                
                ax1.imshow(disp_gt_np, cmap='magma', vmin=0, vmax=dynamic_vmax_gt)
                ax1.set_title(f"GT Disparity (Max: {dynamic_vmax_gt:.1f}px)")
                ax1.axis('off')

            # --- COLUMN 2: CORE PREDICTION ---
            ax2 = axes[row, 1]
            if name == 'yolo':
                yolo_map = det_preds[0][0] 
                obj_score = torch.sigmoid(yolo_map[4, :, :])
                cls_score = torch.sigmoid(yolo_map[5:, :, :]).max(dim=0)[0]
                conf_map = (obj_score * cls_score).cpu().numpy()
                
                # 🚨 Use PowerNorm with gamma=0.3 to violently boost low values!
                # I highly recommend 'turbo' or 'magma' over 'viridis' for spotting speckles
                im2 = ax2.imshow(
                    conf_map, 
                    cmap='turbo', 
                    norm=PowerNorm(gamma=0.3, vmin=0, vmax=1)
                )
                
                ax2.set_title(f"YOLO CONF (Max: {conf_map.max():.2f})")
                plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
                
            else:
                disp_pred_np = disp_pred[0].detach().squeeze().cpu().numpy()
                ax2.imshow(disp_pred_np, cmap='magma', vmin=0, vmax=dynamic_vmax_gt)
                ax2.set_title(f"Predicted Disparity")
                ax2.axis('off')

            # --- COLUMN 3: STUDENT SEGMENTATION ---
            ax3 = axes[row, 2]
            student_seg = torch.argmax(seg_preds, dim=1)[0].cpu().numpy()
            im3 = ax3.imshow(student_seg, cmap='tab20', vmin=0, vmax=10) 
            ax3.set_title("STUDENT PHYSICS SEG")
            if row == 0:
                plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, ticks=range(11))

            # --- COLUMN 4: DEBUG / TEACHER ---
            ax4 = axes[row, 3]
            if name == 'yolo':
                ax4.imshow(img_bg)
                h, w = img_bg.shape[:2]
                yolo_map = det_preds[0][0] # Scale 8
                stride = 8 
                
                # Dekodierung
                obj_score = torch.sigmoid(yolo_map[4, :, :])
                cls_probs = torch.sigmoid(yolo_map[5:, :, :])
                final_conf = obj_score * torch.max(cls_probs, dim=0)[0]
                
                mask = final_conf > VIS_THRESH
                num_candidates = mask.sum().item()
                boxes_drawn = 0
                
                if num_candidates > 0:
                    ys, xs = torch.where(mask)
                    for i in range(len(xs)):
                        gx, gy = xs[i].item(), ys[i].item()
                        
                        dx = torch.sigmoid(yolo_map[0, gy, gx]).item()
                        dy = torch.sigmoid(yolo_map[1, gy, gx]).item()
                        gw = torch.exp(torch.clamp(yolo_map[2, gy, gx], -5, 5)).item()
                        gh = torch.exp(torch.clamp(yolo_map[3, gy, gx], -5, 5)).item()
                        
                        cx_pixel = (gx + dx) * stride
                        cy_pixel = (gy + dy) * stride
                        bw_pixel = gw * stride
                        bh_pixel = gh * stride
                        
                        x1 = cx_pixel - bw_pixel / 2
                        y1 = cy_pixel - bh_pixel / 2
                        
                        cls_id = torch.argmax(cls_probs[:, gy, gx]).item()
                        box_color = plt.cm.tab10(int(cls_id) % 10)
                        
                        alpha = min(1.0, final_conf[gy, gx].item() * 5.0) 

                        rect = plt.Rectangle((x1, y1), bw_pixel, bh_pixel, fill=False, edgecolor=box_color, linewidth=2, alpha=alpha)
                        ax4.add_patch(rect)
                        boxes_drawn += 1
                
                ax4.set_title(f"PRED BOXES (Sigmoid, n={boxes_drawn})")
                
                # 🚨 CLEANUP: Save the results to the explicitly scoped variables!
                yolo_final_conf_max = final_conf.max().item()
                yolo_num_candidates = num_candidates
                yolo_boxes_drawn = boxes_drawn
                has_yolo_stats = True
                
            else:
                with torch.no_grad():
                    teacher_out = teacher_seg(t_img)
                    teacher_physics = torch.argmax(teacher_out, dim=1)[0].cpu().numpy()
                ax4.imshow(teacher_physics, cmap='tab20', vmin=0, vmax=10)
                ax4.set_title("TEACHER TARGET")
                
        for ax in axes.flatten():
            ax.set_xticks([]); ax.set_yticks([])
        
        plt.tight_layout()
        writer.add_figure('Model_Progress/Deep_Vision_Dashboard', fig, global_step=step)
        
        # 🚨 CLEANUP: Only log scalars if YOLO was actually evaluated
        if has_yolo_stats:
            writer.add_scalar('Model_Progress/YOLO_Final_CONF', yolo_final_conf_max, global_step=step)
            writer.add_scalar('Model_Progress/YOLO_Num_Candidates', yolo_num_candidates, global_step=step)
            writer.add_scalar('Model_Progress/YOLO_Boxes_Drawn', yolo_boxes_drawn, global_step=step)
            
        plt.close(fig)
        if was_training: model.train()

print("✅ Visualization updated: YOLO uses Sigmoid-Decoding and Stereo uses Soft-Argmax scale.")

✅ Visualization updated: YOLO uses Sigmoid-Decoding and Stereo uses Soft-Argmax scale.


Cell 7: Model Graph
1. The Shared Backbone: Where the image features are first extracted.
2. The Split: Where the data branches off into three different "heads" (Stereo, Segmentation, and YOLO).
3. The Stereo Neck: How the left and right features are concatenated or subtracted to form a cost volume.
4. Tensor Shapes: Most importantly, it labels the dimensions (e.g., $1 \times 120 \times 160$) at every step, which helps you verify your downsampling logic is working correctly.

In [7]:
# Create a dummy input matching your robot's input size
dummy_left = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)
dummy_right = torch.randn(1, 1, CONFIG['img_height'], CONFIG['img_width']).to(DEVICE)

# Add the graph to TensorBoard
writer = SummaryWriter("./logs")

class GraphWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, l, r):
        # stereo is now the SINGLE final disparity tensor
        stereo, seg, det = self.model(l, r)
        
        # We just return the main outputs we care about visualizing
        # stereo: [B, 1, H, W]
        # seg: [B, 10, H/4, W/4]
        # det: Tuple of lists... let's just grab the first scale for the graph
        
        # Return: Final Disparity, Segmentation, and YOLO Scale 0 (Class+Box)
        return stereo, seg, det[0][0]

# 1. Create the wrapper
wrapper = GraphWrapper(model).to(DEVICE)

# 2. Use the wrapper for the graph
print("📊 Generating model graph for TensorBoard (using wrapper)...")
writer.add_graph(wrapper, [dummy_left, dummy_right])
print("✅ Model graph for TensorBoard done...")
# 3. Clean up memory
del wrapper
torch.cuda.empty_cache()

📊 Generating model graph for TensorBoard (using wrapper)...


/tmp/ipykernel_1054843/3843330303.py:213: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x_left = left.repeat(1, 3, 1, 1) if left.shape[1] == 1 else left
/tmp/ipykernel_1054843/3843330303.py:217: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x_right = right.repeat(1, 3, 1, 1) if right.shape[1] == 1 else right


✅ Model graph for TensorBoard done...


Cell 8: Loss-Wrapper with trainable weights
This class encapsulates FusedHexapodLoss and mangages the weights as trainable parameters.

In [8]:
import torch.nn as nn


def get_balanced_optimizer(model, criterion, base_lr=1e-4):
    # Parameter Gruppen
    backbone_params = model.backbone.parameters()
    yolo_params = model.yolo_head.parameters()
    stereo_params = model.stereo_head.parameters()
    seg_params = model.seg_head.parameters()

    params = [
        # Backbone: Der "Dienstleister" für alle (sehr langsam lernen)
        {'params': backbone_params, 'lr': base_lr * 0.1, 'weight_decay': 1e-5},
        
        # YOLO: Darf schnell konvergieren, aber nicht dominieren
        {'params': yolo_params, 'lr': base_lr * 0.5}, # Reduziert, da Teacher stabilisiert
        
        # Stereo & Seg: Brauchen mehr Zeit für Details
        {'params': stereo_params, 'lr': base_lr * 1.2},
        {'params': seg_params, 'lr': base_lr * 1.0},
        
        # UW Gewichte
        {'params': criterion.parameters(), 'lr': base_lr}
    ]
    return torch.optim.AdamW(params)

Cell 9: The Final Training Loop
This cell sets up and executes the main training process. It includes the crucial hexapod_collate function to handle mixed-data batches, initializes the data loaders, optimizer, and scheduler, and contains the complete training and validation logic with logging to TensorBoard.

In [9]:


# ==============================================================================
# 0. IMPORTANT PRE-REQUISITE (DO THIS BEFORE RUNNING THE LOOP)
# ==============================================================================
# Make sure your FusedHexapodModel is configured for 11 classes!
# If your model currently outputs 10 classes, you MUST update its final layer:
# Example: model.seg_head.classifier = nn.Conv2d(in_channels, 11, kernel_size=1).to(DEVICE)

# ==============================================================================
# 1. GPU-NATIVE TEACHER MODELS
# ==============================================================================
class SegmentationTeacher(nn.Module):
    def __init__(self, device='cuda'):
        super().__init__()
        model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_name)
        self.model.eval()
        self.model.to(device)
        
        for param in self.model.parameters():
            param.requires_grad = False

        self.num_source_classes = 150
        self.num_target_classes = 11  # Updated to 11!
        
        map_matrix = torch.zeros(self.num_source_classes, self.num_target_classes)

        # 0. HARD_FLAT: Floor(3), Wood(19), Road(6), Sidewalk(11), Path(16), Platform(53), Flooring(105)
        map_matrix[[3, 19, 6, 11, 16, 53, 105], 0] = 1.0
        # 1. SOFT_FLAT: Rug(29), Carpet(57)
        map_matrix[[29, 57], 1] = 1.0
        # 2. NATURAL_UNEVEN: Grass(9), Earth(13), Field(28), Sand(46), Soil(94), Land(93), Rock(122)
        map_matrix[[9, 13, 28, 46, 94, 93, 122], 2] = 1.0
        # 3. WATER: Water(21), Sea(26), River(60), Lake(103), Pool(128)
        map_matrix[[21, 26, 60, 103, 128], 3] = 1.0
        # 4. CLUTTER: Pillow(59), Box(70), Paper(96), Towel(100), Clothes(106), Bag(112)
        map_matrix[[59, 70, 96, 100, 106, 112], 4] = 1.0
        # 5. STAIRS: Stairs(25), Step(126), Escalator(124)
        map_matrix[[25, 126, 124], 5] = 1.0
        # 6. STRUCTURAL / BACKGROUND (New Split): Wall(0), Building(1), Fence(32), Railing(104)
        map_matrix[[0, 1, 32, 104], 6] = 1.0
        # 7. FURNITURE / PROPS (New Split): Cabinet(10), Bed(7), Chair(19), Sofa(23), Table(15), Shelf(41)
        map_matrix[[10, 7, 19, 23, 15, 41], 7] = 1.0
        # 8. GLASS: Window(8), Glass(63), Mirror(66)
        map_matrix[[8, 63, 66], 8] = 1.0
        # 9. DYNAMIC: Person(12), Car(20), Bus(80), Bicycle(127)
        map_matrix[[12, 20, 80, 127], 9] = 1.0
        # 10. SKY/VOID: Sky(2), Ceiling(5), Light(82)
        map_matrix[[2, 5, 82], 10] = 1.0

        current_assigned = map_matrix.sum(dim=1)
        unassigned_indices = (current_assigned == 0).nonzero(as_tuple=True)[0]
        map_matrix[unassigned_indices, 6] = 1.0 
        
        self.register_buffer('map_matrix', map_matrix)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def forward(self, x):
        with torch.no_grad():
            x_norm = (x - self.mean.to(x.device)) / self.std.to(x.device)
            outputs = self.model(x_norm)
            probs_150 = F.softmax(outputs.logits, dim=1).permute(0, 2, 3, 1)
            probs_11 = torch.matmul(probs_150, self.map_matrix.to(x.device)).permute(0, 3, 1, 2)
            return torch.log(probs_11 + 1e-6)

class YoloTeacher:
    def __init__(self, device):
        self.model = torch.hub.load('ultralytics/yolov5', 'yolov5m', pretrained=True).to(device).eval()
        self.model.amp = False
        self.device = device
        
        self.conf_thresh = 0.60
        self.iou_thresh = 0.45

    @torch.no_grad()
    def get_pseudo_labels(self, img_tensor):
        """
        100% Pure GPU inference. 
        Bypasses AutoShape, OpenCV, and NumPy to completely eliminate the CPU bottleneck.
        """
        # 1. Un-normalize ImageNet stats ON THE GPU
        mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1, 3, 1, 1)
        
        img_rgb = torch.clamp((img_tensor * std) + mean, 0.0, 1.0) 
        _, _, H, W = img_rgb.shape
        
        # 2. Raw GPU Inference
        outputs = self.model(img_rgb)
        
        # 🚨 FIX: Safely extract the [B, 25200, 85] tensor whether it's wrapped in a tuple or not!
        if isinstance(outputs, (tuple, list)):
            raw_preds = outputs[0]
        else:
            raw_preds = outputs
            
        batch_targets = []
        
        # 3. Blazing Fast GPU NMS
        for i in range(raw_preds.shape[0]):
            pred = raw_preds[i] # Shape: [25200, 85]
            
            # Filter 1: Quick objectness confidence filter
            conf_mask = pred[:, 4] > self.conf_thresh
            pred = pred[conf_mask]
            
            if len(pred) == 0:
                batch_targets.append(torch.empty((0, 5), device=self.device))
                continue
                
            # Compute final class scores (objectness * class_prob)
            cls_scores = pred[:, 5:] * pred[:, 4:5]
            cls_conf, cls_id = cls_scores.max(dim=1, keepdim=True)
            
            # Filter 2: Strict combined confidence filter
            final_mask = cls_conf.squeeze(-1) > self.conf_thresh
            pred = pred[final_mask]
            cls_id = cls_id[final_mask]
            cls_conf = cls_conf[final_mask]
            
            if len(pred) == 0:
                batch_targets.append(torch.empty((0, 5), device=self.device))
                continue
                
            # Extract [cx, cy, w, h] in absolute pixels
            boxes = pred[:, :4]
            
            # Convert to [x1, y1, x2, y2] specifically for torchvision NMS
            x1 = boxes[:, 0] - boxes[:, 2] / 2
            y1 = boxes[:, 1] - boxes[:, 3] / 2
            x2 = boxes[:, 0] + boxes[:, 2] / 2
            y2 = boxes[:, 1] + boxes[:, 3] / 2
            boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=1)
            
            # Class-Aware NMS Trick: Offset boxes by class so different classes don't overlap
            max_coord = boxes_xyxy.max()
            offsets = cls_id.float() * (max_coord + 1)
            boxes_for_nms = boxes_xyxy + offsets
            
            # Run CUDA-optimized NMS
            keep_indices = torchvision.ops.nms(boxes_for_nms, cls_conf.squeeze(-1), self.iou_thresh)
            
            # Normalize boxes to [0, 1] relative to image dimensions
            boxes_norm = boxes[keep_indices].clone()
            boxes_norm[:, 0] /= W  # cx
            boxes_norm[:, 1] /= H  # cy
            boxes_norm[:, 2] /= W  # w
            boxes_norm[:, 3] /= H  # h
            
            classes = cls_id[keep_indices].float()
            
            # Final output format: [class, cx, cy, w, h]
            formatted = torch.cat([classes, boxes_norm], dim=1)
            batch_targets.append(formatted)
            
        return batch_targets

# ==============================================================================
# 2. UTILS & LOSS WRAPPER
# ==============================================================================
def hexapod_collate(batch):
    elem = batch[0]
    collated = {}
    keys_to_stack = ['left', 'right', 'teacher', 'disp', 'seg', 'use_stereo', 'use_seg', 'use_yolo']
    for k in keys_to_stack:
        if k in elem and elem[k] is not None:
            collated[k] = torch.stack([torch.tensor(d[k]) if not isinstance(d[k], torch.Tensor) else d[k] for d in batch])
    if 'det' in elem:
        collated['det'] = [d['det'] for d in batch]
    if 'filename' in elem:
        collated['filename'] = [d['filename'] for d in batch]
    return collated

def find_and_visualize_sample(dataset, filename_part):
    # 1. Dataset-Zugriff (Wrapper umgehen)
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    
    # 2. Suche
    found_idx = -1
    for idx, sample in enumerate(base_ds.samples):
        # Wir suchen nach einem Teilstring im Pfad (z.B. "P001/000456")
        if filename_part in sample['l']:
            found_idx = idx
            break
            
    if found_idx == -1:
        print(f"❌ '{filename_part}' nicht gefunden.")
        return None

    print(f"✅ Gefunden an Index: {found_idx}")
    path = base_ds.samples[found_idx]['l']
    print(f"📂 Pfad: {path}")

    # 3. Daten laden
    s = base_ds.samples[found_idx]
    
    # Bild laden & Check
    img_bgr = cv2.imread(s['l'])
    if img_bgr is None:
        print(f"❌ FEHLER: Konnte Bild nicht laden! Pfad prüfen: {s['l']}")
        return found_idx
        
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Geometrie laden
    geo_data = None
    if s.get('d'):
        try:
            if s['d'].endswith('.pfm'):
                # Stelle sicher, dass read_pfm_fixed im Scope ist!
                geo_data = read_pfm_fixed(s['d']) 
            else:
                geo_data = np.load(s['d'])
        except Exception as e:
            print(f"⚠️ Warnung: Konnte Tiefe nicht laden ({e})")

    # 4. PLOTTING (Explizites Figure-Management)
    # Wir erstellen eine neue Figure, um alte Plots nicht zu überschreiben
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # RGB
    axes[0].imshow(img)
    axes[0].set_title(f"RGB Input\n{os.path.basename(s['l'])}")
    axes[0].axis('off')

    # Depth / Disparity
    if geo_data is not None:
        # Konvertierung für TartanAir (Depth -> Disparity für Visualisierung)
        if s.get('source') == 'tartan':
            # 80.0 ist der Standard-Focal*Baseline Wert für TartanAir
            disp_vis = 80.0 / (geo_data + 1e-6)
            disp_vis = np.clip(disp_vis, 0, 192) # Clipping für Kontrast
            title_str = "Ground Truth (Converted to Disparity)"
        else:
            disp_vis = geo_data
            title_str = "Ground Truth Disparity"

        im = axes[1].imshow(disp_vis, cmap='magma') # 'magma' ist super für Tiefe
        axes[1].set_title(title_str)
        axes[1].axis('off')
        
        # Colorbar hinzufügen
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    else:
        axes[1].text(0.5, 0.5, "Keine GT-Daten vorhanden", ha='center')
        axes[1].axis('off')

    plt.tight_layout()
    plt.show() # Zwingt Jupyter zum Rendern
    
    return found_idx


# Function to pull and prepare a sample
def prepare_static_sample(dataset, idx):
    """
    Zieht ein Sample für den Debugger.
    TRICK: Setzt den Dataset-Modus kurz auf 'eval', um das zufällige 
    Balanced Sampling (COCO-Injektion) zu umgehen.
    """
    # Zugriff auf das originale Full-Dataset
    base_ds = dataset.dataset if hasattr(dataset, 'dataset') else dataset
    
    # 1. Aktuellen Modus sichern und auf 'eval' zwingen
    # Damit deaktivieren wir den "if self.mode == 'train': random..." Block in __getitem__
    prev_mode = base_ds.mode
    base_ds.mode = 'eval'
    
    try:
        # 2. Exaktes Laden des gewünschten Index
        item = base_ds[idx] 
    finally:
        # 3. WICHTIG: Modus sofort wiederherstellen, sonst trainiert er nicht mehr richtig!
        base_ds.mode = prev_mode
    
    # Batch-Dimension hinzufügen (Unsqueeze) und auf GPU schieben
    batch_item = {}
    for k, v in item.items():
        if isinstance(v, torch.Tensor):
            # Tensors auf GPU
            batch_item[k] = v.unsqueeze(0).to(DEVICE)
        elif isinstance(v, (int, float)):
            # Skalare Flags (use_stereo etc.) auch in Tensor wandeln
            batch_item[k] = torch.tensor([v], device=DEVICE)
        else:
            # Listen (det) oder Strings (filename) in eine Liste packen (Pseudo-Batch)
            batch_item[k] = [v]
            
    # Kleiner Print-Check zur Sicherheit
    if 'filename' in batch_item:
        print(f"📦 Loaded Static Batch: {batch_item['filename'][0]}")
        
    return batch_item

class DummyScaler:
    def __init__(self): pass
    def scale(self, loss): return loss
    def step(self, optimizer): optimizer.step()
    def update(self): pass
    def unscale_(self, optimizer): pass

def save_checkpoint(
        model, 
        criterion,   # 🚨 Added the criterion here!
        optimizer, 
        scheduler,
        scaler,
        epoch, 
        loss, 
        filename
    ):
    os.makedirs(CONFIG['save_dir'], exist_ok=True)
    path = os.path.join(CONFIG['save_dir'], filename)
    
    # Safely handle the scheduler if it's None
    sched_state = scheduler.state_dict() if scheduler is not None else None
    
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'criterion_state': criterion.state_dict(), # 🚨 THE CRITICAL ADDITION
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': sched_state,
        'scaler_state': scaler.state_dict(),
        'loss': loss,
    }, path)
    print(f"💾 Checkpoint saved: {path}")

class UncertaintyWeighting(nn.Module):
    def __init__(self, base_criterion):
        super().__init__()
        self.base_criterion = base_criterion
        self.log_var_stereo = nn.Parameter(torch.zeros(1))
        self.log_var_seg = nn.Parameter(torch.zeros(1))
        self.log_var_yolo = nn.Parameter(torch.zeros(1))

    def forward(self, preds, targets, teacher_preds, left_img=None, step=0):
        task_tensors, logs = self.base_criterion(preds, targets, teacher_preds, left_img)
        
        weighted_loss = 0.0
        weighted_logs = {}
        keys = ['stereo', 'seg', 'yolo']
        log_vars_list = [self.log_var_stereo, self.log_var_seg, self.log_var_yolo]

        if step < 1000:
            w = {'stereo': 0.5, 'seg': 1.0, 'yolo': 1.0}
            for key in keys:
                if key in task_tensors:
                    weighted_loss += w[key] * task_tensors[key]
                    weighted_logs[f"weight_{key}"] = w[key]
        else:
            for i, key in enumerate(keys):
                if key not in task_tensors: continue
                # ✅ CHANGE 4a: UW-Clamping (-2.0, 5.0) → (-2.0, 0.7)
                # Gleichgewicht: log_var → log(2*Loss). Mit Loss-Vorskalierung (/10 und /2)
                # liegen alle Tasks bei ~0.5–1.0 → Gleichgewicht bei log_var ≈ 0.0–0.7
                # Clamp bei 0.7 als Sicherheitsnetz gegen explodierendes log_var_yolo
                v = log_vars_list[i].clamp(-2.0, 0.7)
                precision = torch.exp(-v)
                loss = task_tensors[key]

                if key == 'stereo': 
                    weighted_loss += 0.5 * precision * loss + 0.5 * v
                    # 🚨 Fix: Log the true effective weight
                    weighted_logs[f"weight_{key}"] = 0.5 * precision.item() 
                else: 
                    weighted_loss += precision * loss + 0.5 * v
                    weighted_logs[f"weight_{key}"] = precision.item()

        weighted_logs.update(logs)
        return weighted_loss, weighted_logs

# ==============================================================================
# 3. INITIALIZATION & DATA PIPELINE
# ==============================================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Initializing on {DEVICE}...")

# 1. Initialize GPU Teachers in the Main Process
yolo_teacher = YoloTeacher(device=DEVICE)
teacher_seg = SegmentationTeacher(device=DEVICE)

print("📦 Preparing Clean DataLoaders (CPU Multiprocessing Enabled)...")
# REMOVED use_teacher=True and teacher_instance! 
# The dataset now purely loads images and ground truth on the CPU.
tartan_ds = RealFusedDataset(roots={'tartan': '../datasets/TartanAir'}, img_size=(CONFIG['img_height'], CONFIG['img_width']), mode='train')
coco_ds = RealFusedDataset(roots={'coco': '../datasets/coco'}, img_size=(CONFIG['img_height'], CONFIG['img_width']), mode='train')

NEIGHBORHOOD_IDX = find_and_visualize_sample(tartan_ds, "/TartanAir/hospital/Easy/P008/image_left/000163_left.png")
YOLO_IDX = find_and_visualize_sample(coco_ds, "/coco/train2017/000000003145.jpg")
print(NEIGHBORHOOD_IDX,YOLO_IDX)
# Save this for the whole training run
static_batches = {
    'yolo': prepare_static_sample(coco_ds, YOLO_IDX),
    'neighborhood': prepare_static_sample(tartan_ds, NEIGHBORHOOD_IDX)
}
# Because the Dataset is pure CPU, you can safely use your desired num_workers!
tartan_loader = DataLoader(tartan_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True, drop_last=True, collate_fn=hexapod_collate)
coco_loader = DataLoader(coco_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True, drop_last=True, collate_fn=hexapod_collate)

criterion_base = FusedHexapodLoss(CONFIG)
criterion = UncertaintyWeighting(criterion_base).to(DEVICE)

model.requires_grad_(False)
model.backbone.requires_grad_(True)
model.stereo_head.requires_grad_(True)
model.seg_head.requires_grad_(True)
model.yolo_head.requires_grad_(True)
criterion.requires_grad_(True) 
model.backbone.eval() # BN LOCKED

# 🚨 Boosted base_lr: MobileNetV3 and fresh heads can comfortably handle this!
base_lr = 5e-4  

optimizer = torch.optim.AdamW([
    # Backbone: Pretrained MobileNetV3. Needs to learn, but gently. (Peak: 5e-5)
    {'params': model.backbone.parameters(),  'lr': base_lr * 0.1, 'weight_decay': 1e-5, 'name': 'Backbone'}, 
    
    # 🚨 YOLO Head: Boosted from 0.5 to 1.0! It needs energy to initialize anchors/biases. (Peak: 5e-4)
    {'params': model.yolo_head.parameters(), 'lr': base_lr * 1.0, 'name': 'Yolo_Head'}, 
    
    # Stereo Head: Heavy 3D cost volumes need massive gradients. (Peak: 1e-3)
    # ✅ CHANGE 4b: Stereo LR 2.0→1.0 (war zu aggressiv, UW kompensiert)
    {'params': model.stereo_head.parameters(), 'lr': base_lr * 1.0, 'name': 'Stereo_Head'},  
    
    # Seg Head: Standard distillation mapping. (Peak: 5e-4)
    {'params': model.seg_head.parameters(),  'lr': base_lr * 1.0, 'name': 'Seg_Head'}, 
    
    # UW Weights: Slowed down relative to base_lr to prevent erratic task flipping. (Peak: 1e-4)
    # ✅ CHANGE 4c: UW LR 0.2→0.5 (muss schneller reagieren als Heads)
    {'params': criterion.parameters(),       'lr': base_lr * 0.5, 'name': 'Uncertainty_Weights'}
], weight_decay=1e-4)

steps_per_epoch = len(coco_loader) // CONFIG['ACCUMULATION_STEPS']

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[
        base_lr * 0.1,  # Backbone
        base_lr * 1.0,  # YOLO
        base_lr * 2.0,  # Stereo
        base_lr * 1.0,  # Seg
        base_lr * 0.2   # UW
    ],
    epochs=CONFIG['num_epochs'],
    steps_per_epoch=steps_per_epoch,
    # ✅ CHANGE 4d: pct_start 0.3→0.1 (Peak früher, mehr Zeit zum Konvergieren)
    pct_start=0.1,
    div_factor=25.0,    # Starts at peak / 25
    final_div_factor=1000.0 # Finishes tiny for ultra-fine tuning
)



    
def infinite_loader(dataloader):
    """Yields batches indefinitely without caching them in memory."""
    while True:
        for batch in dataloader:
            yield batch



🚀 Initializing on cuda...


Using cache found in /home/slarc/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-2-11 Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 12288MiB)

Fusing layers... 
YOLOv5m summary: 290 layers, 21172173 parameters, 0 gradients, 48.9 GFLOPs
Adding AutoShape... 
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


📦 Preparing Clean DataLoaders (CPU Multiprocessing Enabled)...
   Scanning TartanAir...
   Scanning TartanAir...
   -> Found 84824 TartanAir samples.
   Scanning COCO...
   Scanning COCO 2017...
loading annotations into memory...
Done (t=10.46s)
creating index...
index created!
   -> Found 117266 COCO samples.
✅ Gefunden an Index: 67803
📂 Pfad: ../datasets/TartanAir/hospital/Easy/P008/image_left/000163_left.png
✅ Gefunden an Index: 30976
📂 Pfad: ../datasets/coco/train2017/000000003145.jpg
67803 30976


==============================================================================
6. RUN Phase 2 Refinement
==============================================================================

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RefinementYOLOLoss(SimpleYOLOLoss):
    """
    Phase 2 YOLO Loss: Nutzt IoU-Aware Objectness und GIoU für scharfe Bounding Boxen.
    Gibt die IoU-Werte zurück, damit wir sie in TensorBoard plotten können.
    """
    def forward(self, preds, targets, global_step=0):
        reg_p = preds[:, :4, :, :]   
        obj_p = preds[:, 4:5, :, :]  
        cls_p = preds[:, 5:, :, :]   

        cls_t, reg_t, obj_mask = self._get_targets(targets, cls_p, reg_p)
        
        num_targets_val = obj_mask.sum().item()
        max_cls_val = cls_t.max().item() if num_targets_val > 0 else 0
        num_pos = torch.clamp(obj_mask.sum(), min=1.0)

        if num_targets_val == 0:
            return {
                'total': torch.tensor(0.0, requires_grad=True, device=preds.device),
                'box': 0.0, 'obj': 0.0, 'cls': 0.0, 'num_targets': 0, 'max_cls': 0,
                'ious': torch.tensor([], device=preds.device)
            }

        # --- THE IOU & GIOU UPGRADE BLOCK ---
        b, _, cy, cx = torch.where(obj_mask > 0)
        obj_target = obj_mask.clone()
        l_giou = torch.tensor(0.0, device=preds.device)

        if len(b) > 0:
            # Decode Predictions & Targets (mit Clamp für numerische Stabilität)
            p_x = reg_p[b, 0, cy, cx] + cx
            p_y = reg_p[b, 1, cy, cx] + cy
            p_w = torch.exp(torch.clamp(reg_p[b, 2, cy, cx], max=5.0))
            p_h = torch.exp(torch.clamp(reg_p[b, 3, cy, cx], max=5.0))
            
            t_x = reg_t[b, 0, cy, cx] + cx
            t_y = reg_t[b, 1, cy, cx] + cy
            t_w = torch.exp(torch.clamp(reg_t[b, 2, cy, cx], max=5.0))
            t_h = torch.exp(torch.clamp(reg_t[b, 3, cy, cx], max=5.0))

            # Convert to Corners
            px1, px2 = p_x - p_w / 2, p_x + p_w / 2
            py1, py2 = p_y - p_h / 2, p_y + p_h / 2
            tx1, tx2 = t_x - t_w / 2, t_x + t_w / 2
            ty1, ty2 = t_y - t_h / 2, t_y + t_h / 2

            # Intersection Area
            inter_x1, inter_y1 = torch.max(px1, tx1), torch.max(py1, ty1)
            inter_x2, inter_y2 = torch.min(px2, tx2), torch.min(py2, ty2)
            inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)

            # Union Area & IoU
            union_area = (p_w * p_h) + (t_w * t_h) - inter_area + 1e-6
            iou = inter_area / union_area
            
            # GIoU (Generalized IoU) with Convex Hull
            hull_x1, hull_y1 = torch.min(px1, tx1), torch.min(py1, ty1)
            hull_x2, hull_y2 = torch.max(px2, tx2), torch.max(py2, ty2)
            hull_area = torch.clamp(hull_x2 - hull_x1, min=0) * torch.clamp(hull_y2 - hull_y1, min=0) + 1e-6
            
            giou = iou - (hull_area - union_area) / hull_area
            l_giou = (1.0 - giou).sum() / num_pos 

            # Objectness Target: binär → IoU-Aware über 10000 Steps
            iou_aware_weight = min(1.0, global_step / 10000)
            obj_target[b, 0, cy, cx] = (
                iou_aware_weight * iou.detach().clamp(0.35, 1.0) +
                (1.0 - iou_aware_weight) * 1.0
            ).type(obj_target.dtype)

        # Focal Loss mit höherem alpha
        l_obj = self.sigmoid_focal_loss(obj_p, obj_target, alpha=0.85, reduction='sum') / num_pos

        # 3. REGRESSION LOSS (L1 + heavy GIoU)
        l_offset = (self.l1(reg_p[:, :2], reg_t[:, :2]) * obj_mask).sum() / num_pos
        l_size = (self.l1(reg_p[:, 2:], reg_t[:, 2:]) * obj_mask).sum() / num_pos
        # GIoU Warm-up: erst ab Step 10000 voll aktiv
        giou_weight = min(0.5, global_step / 10000)
        l_box = l_offset + 2.0 * l_size + giou_weight * l_giou

        # 4. CLASSIFICATION LOSS
        l_cls_raw = self.sigmoid_focal_loss(cls_p, cls_t, reduction='none')
        l_cls = (l_cls_raw * obj_mask).sum() / num_pos

        return {
            'total': 5.0 * l_box + 5.0 * l_obj + l_cls, 
            'box': l_box.item(), 'obj': l_obj.item(), 'cls': l_cls.item(),
            'num_targets': num_targets_val, 'max_cls': max_cls_val,
            'ious': iou.detach() if len(b) > 0 else torch.tensor([], device=preds.device)
        }

class SobelGradientLoss(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1, 0, 1],
                            [-2, 0, 2],
                            [-1, 0, 1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],
                            [ 0, 0, 0],
                            [ 1, 2, 1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def forward(self, pred_disp, gt_disp, mask):
        pred = pred_disp * mask
        gt   = gt_disp   * mask
        pred_gx = F.conv2d(pred, self.kx, padding=1)
        pred_gy = F.conv2d(pred, self.ky, padding=1)
        gt_gx   = F.conv2d(gt,   self.kx, padding=1)
        gt_gy   = F.conv2d(gt,   self.ky, padding=1)
        return (F.l1_loss(pred_gx[mask], gt_gx[mask])
              + F.l1_loss(pred_gy[mask], gt_gy[mask])) * 0.5


class LaplacianLoss(nn.Module):
    def __init__(self):
        super().__init__()
        k = torch.tensor([[0,  1, 0],
                           [1, -4, 1],
                           [0,  1, 0]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('k', k)

    def forward(self, pred_disp, gt_disp, mask):
        lap_pred = F.conv2d(pred_disp, self.k, padding=1)
        lap_gt   = F.conv2d(gt_disp,   self.k, padding=1)
        return F.l1_loss(lap_pred[mask], lap_gt[mask])

class RefinementHexapodLoss(nn.Module):
    """
    Phase 2 Unified Loss: Pure L1 Stereo for sharp edges, static weights, and IoU YOLO.
    """
    def __init__(self, config):
        super().__init__()
        self.yolo_s8_loss   = RefinementYOLOLoss(config['num_det_classes'], stride=8)
        self.yolo_s16_loss  = RefinementYOLOLoss(config['num_det_classes'], stride=16)
        self.yolo_s32_loss  = RefinementYOLOLoss(config['num_det_classes'], stride=32)
        self.smooth_loss    = EdgeAwareSmoothnessLoss()
        self.sobel_loss     = SobelGradientLoss()    # ✅ NEU
        self.laplacian_loss = LaplacianLoss()         # ✅ NEU
        self.register_buffer('log_var_stereo', torch.tensor(0.0))
        self.register_buffer('log_var_seg',    torch.tensor(0.0))
        self.register_buffer('log_var_yolo',   torch.tensor(0.0))

    def forward(self, preds, targets, teacher_preds=None, left_img=None, coarse_disp=None, global_step=0):
        stereo_preds, seg_preds, det_preds = preds
        task_tensors, logs = {}, {}

        logs['weight_stereo'] = 1.0
        logs['weight_yolo']   = 1.0
        logs['weight_seg']    = 0.0

        # --- 1. STEREO LOSS ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                l_stereo_raw = F.l1_loss(stereo_preds[mask], gt_disp[mask])

                if left_img is not None:
                    l_smooth = self.smooth_loss(stereo_preds, left_img)
                    dynamic_smooth_weight = torch.clamp(l_stereo_raw.detach() * 0.10, max=0.25)
                    l_stereo = l_stereo_raw + dynamic_smooth_weight * l_smooth
                    logs['smooth'] = l_smooth.item()
                else:
                    l_stereo = l_stereo_raw

                # Aux-Loss (s8-Skala)
                if coarse_disp is not None:
                    gt_s8 = F.interpolate(gt_disp, size=coarse_disp.shape[-2:],
                                        mode='nearest') / 8.0
                    mask_s8 = (gt_s8 > 0) & (gt_s8 < 24)
                    if mask_s8.sum() > 0:
                        l_aux    = F.l1_loss(coarse_disp[mask_s8], gt_s8[mask_s8])
                        l_stereo = l_stereo + 0.1 * l_aux   # ✅ 0.3→0.1
                        logs['stereo_aux'] = l_aux.item()

                # ✅ NEU: Sobel + Laplacian Schärfe-Losses
                l_sobel     = self.sobel_loss(stereo_preds, gt_disp, mask)
                l_laplacian = self.laplacian_loss(stereo_preds, gt_disp, mask)
                l_stereo    = l_stereo + 0.05 * l_sobel + 0.02 * l_laplacian
                logs['stereo_sobel']     = l_sobel.item()
                logs['stereo_laplacian'] = l_laplacian.item()

                # ✅ NEU: Gradient-Skalierung /2 — analog zu Phase 1
                # Stereo Pre-Clip Gradient war bis 305, YOLO bei ~5
                # /2 halbiert den Gradienten VOR dem Clipping → Clipping feuert seltener
                task_tensors['stereo'] = l_stereo / 4.0
                logs['stereo'] = l_stereo_raw.item()  # raw ohne Skalierung loggen

        # ✅ NEU: KD-Loss für Seg (nur wenn teacher_preds übergeben werden, also Phase-2b)
        if seg_preds is not None and teacher_preds is not None:
            T = 2.0
            s_logits = F.interpolate(seg_preds, size=teacher_preds.shape[-2:],
                                    mode='bilinear', align_corners=False)
            p_s = F.log_softmax(s_logits / T, dim=1)
            p_t = F.softmax(teacher_preds / T, dim=1)
            kl_loss_pixelwise = F.kl_div(p_s, p_t, reduction='none') * (T ** 2)
            l_kd = kl_loss_pixelwise.sum(dim=1).mean()
            task_tensors['seg'] = l_kd
            logs['seg_kd'] = l_kd.item()
            logs['weight_seg'] = 1.0

        # --- 2. YOLO LOSS ---
        if det_preds is not None and targets.get('det') is not None:
            gt_det = targets['det']
            r8  = self.yolo_s8_loss(det_preds[0],  gt_det, global_step=global_step)
            r16 = self.yolo_s16_loss(det_preds[1], gt_det, global_step=global_step)
            r32 = self.yolo_s32_loss(det_preds[2], gt_det, global_step=global_step)
            l_yolo = (r8['total'] + r16['total'] + r32['total']) / 3.0
            task_tensors['yolo'] = l_yolo
            logs['yolo'] = l_yolo.item()
            def _to_float(v): return v.item() if isinstance(v, torch.Tensor) else float(v)
            logs['yolo_box'] = _to_float((r8['box'] + r16['box'] + r32['box']) / 3.0)
            logs['yolo_obj'] = _to_float((r8['obj'] + r16['obj'] + r32['obj']) / 3.0)
            logs['yolo_cls'] = _to_float((r8['cls'] + r16['cls'] + r32['cls']) / 3.0)
            logs['yolo_num_targets'] = r8.get('num_targets', 0) + r16.get('num_targets', 0) + r32.get('num_targets', 0)
            logs['yolo_max_cls']     = max(r8.get('max_cls', 0), r16.get('max_cls', 0), r32.get('max_cls', 0))
            if len(r8['ious']) > 0 or len(r16['ious']) > 0 or len(r32['ious']) > 0:
                logs['step_ious'] = torch.cat([i for i in [r8['ious'], r16['ious'], r32['ious']] if len(i) > 0])

        total_loss = 0.0
        if 'stereo' in task_tensors: total_loss += 1.0 * task_tensors['stereo']
        if 'seg'    in task_tensors: total_loss += 0.5 * task_tensors['seg']    # ← 0.5 damit Stereo/YOLO nicht verdrängt werden
        if 'yolo'   in task_tensors: total_loss += 1.0 * task_tensors['yolo']

        return total_loss, logs

In [11]:
print("--- PREPARING PHASE 2: REFINEMENT ---")

# 1. Load Checkpoint (angepasst an deine Save-Logik und PyTorch Sicherheit)
checkpoint = torch.load(
    './checkpoints/checkpoint_epoch_24.pth', 
    map_location=DEVICE, 
    weights_only=True # 🚨 Behebt die FutureWarning
)

# ✅ CHANGE 5a: strict=False → neue Parameter (context, mid_classifier)
# werden zufällig initialisiert; alle anderen Gewichte aus Checkpoint geladen.
missing, unexpected = model.load_state_dict(checkpoint['model_state'], strict=False)
print(f"  Neu initialisiert (nicht im Checkpoint): {missing}")
print(f"  Ignoriert (nicht im Modell):             {unexpected}")

# ✅ CHANGE 5b: Identity-Init für das Context Network
# Ohne Init: zufällige Gewichte → Cost Volume wird sofort verzerrt → Loss-Spike
# Mit Identity-Init: context(x) ≈ x beim ersten Schritt → kein Unterschied zu Phase 1
# Das Netz lernt dann schrittweise Abweichungen von der Identität.
with torch.no_grad():
    # Depthwise: dirac_ erzeugt Einheitsimpuls → jede Ebene wird unverändert durchgereicht
    nn.init.dirac_(model.stereo_head.context[0].weight)
    # Pointwise: Einheitsmatrix → Ebenen werden nicht gemischt
    context_pw_w = model.stereo_head.context[1].weight  # [max_disp, max_disp, 1, 1]
    nn.init.eye_(context_pw_w.squeeze(-1).squeeze(-1))
    model.stereo_head.context[1].bias.zero_()
print("✅ Context Network Identity-Init applied (kein Loss-Spike garantiert).")

#print("🔧 Resetting obj_pred layers for IoU-Aware Objectness...")
#for m in model.yolo_head.modules():
#    if isinstance(m, DecoupledHead):
#        nn.init.normal_(m.obj_pred.weight, std=0.01)
#        nn.init.constant_(m.obj_pred.bias, -2.0)
#print("✅ obj_pred reset complete.")

# Namensänderung für Kompatibilität mit deinem Log-Block unten
criterion = RefinementHexapodLoss(CONFIG).to(DEVICE)

# 2. Freezing Strategie
for param in model.seg_head.parameters():
    param.requires_grad = False

def set_backbone_freeze(model, freeze_until_layer=4):
    for name, param in model.backbone.named_parameters():
        if 'blocks' in name:
            try:
                layer_idx = int(name.split('blocks.')[1].split('.')[0])
                param.requires_grad = (layer_idx >= freeze_until_layer)
            except:
                pass
    frozen    = sorted(set(int(n.split('blocks.')[1].split('.')[0])
                  for n,p in model.backbone.named_parameters()
                  if 'blocks' in n and not p.requires_grad))
    trainable = sorted(set(int(n.split('blocks.')[1].split('.')[0])
                  for n,p in model.backbone.named_parameters()
                  if 'blocks' in n and p.requires_grad))
    print(f"  Frozen blocks:    {frozen}")
    print(f"  Trainable blocks: {trainable}")

# Wir starten sicher: Nur tiefe Schichten (ab Layer 8) liefern Gradienten für die Heads
set_backbone_freeze(model, freeze_until_layer=4)

# 3. Setup Optimizer
base_lr_p2 = 5e-5

optimizer = torch.optim.AdamW([
    {'params': filter(lambda p: p.requires_grad, model.backbone.parameters()), 'lr': base_lr_p2 * 0.1, 'name': 'Backbone'},
    {'params': model.yolo_head.parameters(),  'lr': base_lr_p2 * 3.0, 'name': 'Yolo_Head'},  # war 2.0
    {'params': model.stereo_head.parameters(),'lr': base_lr_p2 * 1.5, 'name': 'Stereo_Head'},
    {'params': model.seg_head.parameters(),   'lr': base_lr_p2 * 0.5, 'name': 'Seg_Head'},   # ← NEU
], weight_decay=1e-4)

# 4. Cosine Annealing 
steps_per_epoch_p2 = len(coco_loader) // CONFIG['ACCUMULATION_STEPS']
refinement_epochs = 12 # 12 Epochen sind ideal für diese Phase

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=refinement_epochs * steps_per_epoch_p2, 
    eta_min=1e-6
)

print(f"Phase 2 Setup Complete. Refinement for {refinement_epochs} epochs.")

--- PREPARING PHASE 2: REFINEMENT ---
  Neu initialisiert (nicht im Checkpoint): ['stereo_head.context.0.weight', 'stereo_head.context.1.weight', 'stereo_head.context.1.bias', 'seg_head.mid_classifier.weight', 'seg_head.mid_classifier.bias']
  Ignoriert (nicht im Modell):             []
✅ Context Network Identity-Init applied (kein Loss-Spike garantiert).
  Frozen blocks:    [0, 1, 2, 3]
  Trainable blocks: [4, 5, 6]
Phase 2 Setup Complete. Refinement for 12 epochs.


In [12]:
# ====================================================================
# 📸 PREPARE STATIC DASHBOARD BATCHES (Run once before training!)
# ====================================================================
from torch.utils.data.dataloader import default_collate

print("📸 Extracting specific static validation batches for the dashboard...")

def get_batch_by_filename(dataset, target_filename, path_attribute='left_paths'):
    target_idx = 0  
    
    if hasattr(dataset, path_attribute):
        paths = getattr(dataset, path_attribute)
        for i, path in enumerate(paths):
            if target_filename in str(path):
                target_idx = i
                print(f"✅ Bild '{target_filename}' gefunden an Index {i}.")
                break
        else:
            print(f"⚠️ Bild '{target_filename}' nicht gefunden. Nutze Fallback (Index 0).")
    else:
        print(f"⚠️ Dataset hat kein Attribut '{path_attribute}'. Nutze Fallback (Index 0).")

    # 1. Das einzelne Item über den Index laden
    single_item = dataset[target_idx]
    
    # 2. 🚨 DEINE EIGENE Funktion nutzen anstatt default_collate!
    batch = hexapod_collate([single_item])
    
    return batch

# 🎯 DEINE BILDER
TARTAN_TARGET_FILE = "/TartanAir/office2/Easy/P000/image_left/000336_left.png" 
COCO_TARGET_FILE = "/coco/train2017/000000003145.jpg"

# Abruf direkt aus dem Dataset des Loaders ('.dataset' Trick)
# (In deinem Datensatz heißen die Listen 'samples', und darin der Schlüssel 'l')
# Ich passe das direkt so an, dass es für deinen Code out-of-the-box klappt:
def get_batch_from_samples(dataset, target_filename):
    for i, sample in enumerate(dataset.samples):
        if target_filename in sample['l']:
            print(f"✅ Bild '{target_filename}' gefunden an Index {i}.")
            return hexapod_collate([dataset[i]])
    print(f"⚠️ Bild '{target_filename}' nicht gefunden. Nutze Fallback (Index 0).")
    return hexapod_collate([dataset[0]])

tartan_val = get_batch_from_samples(tartan_loader.dataset, TARTAN_TARGET_FILE)
coco_val = get_batch_from_samples(coco_loader.dataset, COCO_TARGET_FILE)

# 3. Auf die GPU schieben (für dein Dashboard)
with torch.no_grad():
    t_l_val = tartan_val['left'].to(DEVICE)
    t_teacher_img_val = tartan_val['teacher'].to(DEVICE)
    c_l_val = coco_val['left'].to(DEVICE)
    
    t_seg_teacher_val = teacher_seg(t_teacher_img_val)
    t_det_pseudo_val = yolo_teacher.get_pseudo_labels(t_l_val)
    
    if 'det' in coco_val and coco_val['det'][0] is not None:
        c_det_target_val = [t.to(DEVICE) for t in coco_val['det']]
    else:
        c_det_target_val = yolo_teacher.get_pseudo_labels(c_l_val)

# 3. Dictionaries für die Visualisierung packen
tartan_viz = dict(tartan_val)
tartan_viz['det'] = t_det_pseudo_val
tartan_viz['seg'] = t_seg_teacher_val.argmax(dim=1) 

coco_viz = dict(coco_val)
coco_viz['det'] = c_det_target_val

# 4. Globale Variable für das Dashboard setzen
global static_batches
static_batches = {
    'tartan': tartan_viz,
    'yolo': coco_viz
}

# ====================================================================
# 🎛️ INITIALIZE TRACKERS & SCALER
# ====================================================================
# --- Smart Visualization Trackers ---
best_viz_loss = float('inf')
last_viz_step = 0
smoothed_loss = None

MIN_VIZ_STEPS = 100       
MAX_VIZ_STEPS = 2000      
LOSS_DROP_THRESH = 0.05   

# 🚨 HIER IST DER FEHLENDE SCALER!
scaler = torch.cuda.amp.GradScaler()

print("✅ Scaler und Visualisierungs-Variablen erfolgreich für Phase 2 geladen!")

📸 Extracting specific static validation batches for the dashboard...
✅ Bild '/TartanAir/office2/Easy/P000/image_left/000336_left.png' gefunden an Index 62258.
✅ Bild '/coco/train2017/000000003145.jpg' gefunden an Index 30976.
✅ Scaler und Visualisierungs-Variablen erfolgreich für Phase 2 geladen!


/tmp/ipykernel_1054843/4018225704.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [13]:
import matplotlib.pyplot as plt
import io
from PIL import Image
import torchvision.transforms.functional as TF

def log_iou_distribution(writer, ious, global_step, tag='Phase2_Debug/IoU_Histogram'):
    """
    Erstellt ein Histogramm der IoU-Werte und sendet es an TensorBoard.
    
    Parameter:
    * writer: Dein TensorBoard SummaryWriter.
    * ious: Ein 1D-Tensor mit den gesammelten IoU-Werten.
    * global_step: Der aktuelle Trainingsschritt.
    """
    # Überspringen, falls keine Boxen gefunden wurden
    if ious is None or len(ious) == 0:
        return

    # Auf die CPU schieben und in Numpy umwandeln
    if isinstance(ious, torch.Tensor):
        ious = ious.detach().cpu().numpy()

    # Matplotlib Figure erstellen
    fig, ax = plt.subplots(figsize=(6, 4))
    
    # Histogramm zeichnen (Grün = Erfolg, je weiter rechts desto besser)
    ax.hist(ious, bins=20, range=(0.0, 1.0), color='mediumseagreen', edgecolor='black', alpha=0.7)
    
    ax.set_title(f"YOLO Box IoU Verteilung (Step {global_step})")
    ax.set_xlabel("IoU Wert (1.0 = Perfekte Überlappung)")
    ax.set_ylabel("Anzahl der Boxen")
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Plot in einen virtuellen Puffer speichern, um ihn für TensorBoard in ein Bild zu verwandeln
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig) # Schließen, um Speicherlecks zu vermeiden!
    buf.seek(0)

    # Bild laden und in einen Tensor konvertieren
    img = Image.open(buf)
    img_tensor = TF.to_tensor(img)
    
    # Ab zu TensorBoard!
    writer.add_image(tag, img_tensor, global_step)

def get_grad_norm(parameters):
    """Berechnet die L2-Norm der Gradienten (vor dem Clipping)."""
    total_norm = 0.0
    for p in parameters:
        if p.grad is not None:
            param_norm = p.grad.detach().data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

In [ ]:
# Füge das vor deiner Phase-2-Schleife ein:
import gc

# 1. Alte Iteratoren löschen (falls vorhanden)
if 'tartan_iter' in locals(): del tartan_iter
if 'coco_iter' in locals(): del coco_iter
gc.collect()

# 2. DataLoader GANZ NEU initialisieren (ersetze dies mit deinem tatsächlichen Code zur Erstellung!)
# tartan_loader = DataLoader(tartan_dataset, batch_size=..., num_workers=...) 
# coco_loader = DataLoader(coco_dataset, batch_size=..., num_workers=...)
tartan_loader = DataLoader(tartan_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True, drop_last=True, collate_fn=hexapod_collate)
coco_loader = DataLoader(coco_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True, drop_last=True, collate_fn=hexapod_collate)


def train_phase2_one_epoch(epoch_idx, tartan_loader, coco_loader):
    global smoothed_loss, best_viz_loss, last_viz_step, global_step, scaler
    
    # ✅ CHANGE 6a: Phase-2b Übergang
    if epoch_idx == CONFIG['PHASE2B_EPOCH']:
        print(f"\n🔓 PHASE-2b START (Epoch {epoch_idx}): Backbone tiefer auftauen + Seg unfreezen + Dilated Conv aktivieren")

        # Backbone: mittlere Layer auftauen (ab Layer 4 statt 8)
        set_backbone_freeze(model, freeze_until_layer=2)

        # ✅ Seg Head auftauen
        for param in model.seg_head.parameters():
            param.requires_grad = True
        print("  ✅ Seg Head: aufgetaut")

        # ✅ Dilated Conv (mid_classifier) aktivieren
        model.seg_head.use_mid = True
        print("  ✅ Dilated Conv (mid_classifier, dilation=2): aktiviert")
        
        # KORREKTUR: Optimizer & Scheduler komplett neu initialisieren, 
        # um Momentum-Verlust oder Abstürze in AdamW zu vermeiden!
        global optimizer, scheduler
        optimizer = torch.optim.AdamW([
            {'params': filter(lambda p: p.requires_grad, model.backbone.parameters()), 'lr': base_lr_p2 * 0.1, 'name': 'Backbone'},
            {'params': model.yolo_head.parameters(),  'lr': base_lr_p2 * 3.0, 'name': 'Yolo_Head'},
            {'params': model.stereo_head.parameters(), 'lr': base_lr_p2 * 1.5, 'name': 'Stereo_Head'},
            {'params': model.seg_head.parameters(),   'lr': base_lr_p2 * 0.5, 'name': 'Seg_Head'},  # ← NEU
        ], weight_decay=1e-4)

        
        # Scheduler für die restlichen Epochen anpassen
        remaining_epochs = refinement_epochs - 6
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=remaining_epochs * steps_per_epoch_p2, 
            eta_min=1e-6
        )

    model.train()
    model.backbone.eval() # BN locked halten!
    criterion.train()
    
    running_loss = 0.0
    tartan_iter = iter(infinite_loader(tartan_loader))
    coco_iter = iter(infinite_loader(coco_loader))
    
    pbar = tqdm(range(steps_per_epoch_p2), desc=f"Phase 2 - Epoch {epoch_idx+1}/{refinement_epochs}")
    all_step_ious = [] # Speicher für das IoU Histogramm

    for step in pbar:
        global_step = epoch_idx * steps_per_epoch_p2 + step
        total_step_loss = 0.0
        step_logs = {}

        optimizer.zero_grad(set_to_none=True)

        # ==========================================
        # --- PHASE 2: TARTANAIR (STEREO + YOLO) ---
        # ==========================================
        for _ in range(CONFIG['ACCUMULATION_STEPS']):
            batch = next(tartan_iter)
            l, r = batch['left'].to(DEVICE), batch['right'].to(DEVICE)
            targets = {'disp': batch['disp'].to(DEVICE), 'det': None}

            with torch.cuda.amp.autocast():
                features_l = model.backbone(l)
                features_r = model.backbone(r)
                l_img_gray = l.mean(dim=1, keepdim=True)

                d_raw, coarse_disp = model.stereo_head(l_s8=features_l[1], r_s8=features_r[1],
                                        l_s4=features_l[0], l_img_raw=l_img_gray)
                d_raw = d_raw.float()
                coarse_disp = coarse_disp.float()

                # ✅ NEU: seg_head aufrufen (Phase-2a: eingefroren → kein Gradient, trotzdem kein Crash)
                seg_raw = model.seg_head(features_l[0], features_l[2])

            d_raw   = d_raw.float()
            seg_raw = seg_raw.float()

            # Teacher-Preds nur in Phase-2b (wenn seg_head trainierbar)
            teacher_preds = None
            if model.seg_head.use_mid:  # True ab Epoch 6
                with torch.no_grad():
                    teacher_preds = teacher_seg(batch['teacher'].to(DEVICE))

            loss, logs = criterion((d_raw, seg_raw, None), targets,
                        teacher_preds=teacher_preds, left_img=l,
                        coarse_disp=coarse_disp, global_step=global_step) 
            loss = loss / (2.0 * CONFIG['ACCUMULATION_STEPS'])

            scaler.scale(loss).backward()
            total_step_loss += loss.item()
            for k, v in logs.items():
                if k == 'step_ious': all_step_ious.append(v)
                else: step_logs[k] = step_logs.get(k, 0) + (v / CONFIG['ACCUMULATION_STEPS'])
                
        # ==========================================
        # --- PHASE 2: COCO (YOLO ONLY) ---
        # ==========================================
        for _ in range(CONFIG['ACCUMULATION_STEPS']):
            batch = next(coco_iter)
            l = batch['left'].to(DEVICE)
            
            if 'det' in batch and batch['det'][0] is not None:
                coco_det = [t.to(DEVICE) for t in batch['det']]
            else:
                with torch.no_grad(): coco_det = yolo_teacher.get_pseudo_labels(l)

            targets = {'disp': None, 'det': coco_det}

            with torch.cuda.amp.autocast():
                features_l = model.backbone(l)
                y_raw_coco = model.yolo_head(x_s8=features_l[1], x_s16=features_l[2], x_s32=features_l[3])
            
            y_raw_coco = [y.float() for y in y_raw_coco]
            
            loss, logs = criterion((None, None, y_raw_coco), targets, left_img=l, global_step=global_step)
            loss = loss / (2.0 * CONFIG['ACCUMULATION_STEPS'])
            
            scaler.scale(loss).backward() 
            total_step_loss += loss.item()
            for k, v in logs.items(): 
                if k == 'step_ious': all_step_ious.append(v)
                else: step_logs[k] = step_logs.get(k, 0) + (v / CONFIG['ACCUMULATION_STEPS'])

        # ==========================================
        # --- STEP & LOGGING ---
        # ==========================================
        scaler.unscale_(optimizer)

        # 📊 MESSUNG 1: Gradienten VOR dem Clipping (Rohzustand)
        step_logs['Grad_Norm_Pre/Backbone'] = get_grad_norm(model.backbone.parameters())
        step_logs['Grad_Norm_Pre/Stereo'] = get_grad_norm(model.stereo_head.parameters())
        step_logs['Grad_Norm_Pre/Seg'] = get_grad_norm(model.seg_head.parameters())
        step_logs['Grad_Norm_Pre/Yolo'] = get_grad_norm(model.yolo_head.parameters())

        # 2. Das Clipping anwenden
        torch.nn.utils.clip_grad_norm_(model.stereo_head.parameters(), max_norm=10.0)
        torch.nn.utils.clip_grad_norm_(model.yolo_head.parameters(),   max_norm=5.0)
        torch.nn.utils.clip_grad_norm_(model.backbone.parameters(),    max_norm=1.0)  # war 0.5
        torch.nn.utils.clip_grad_norm_(model.seg_head.parameters(),    max_norm=2.0)  # bleibt
        

        # 📊 MESSUNG 2: Gradienten NACH dem Clipping (Was der Optimizer wirklich bekommt)
        step_logs['Grad_Norm_Post/Backbone'] = get_grad_norm(model.backbone.parameters())
        step_logs['Grad_Norm_Post/Stereo'] = get_grad_norm(model.stereo_head.parameters())
        step_logs['Grad_Norm_Post/Seg'] = get_grad_norm(model.seg_head.parameters())
        step_logs['Grad_Norm_Post/Yolo'] = get_grad_norm(model.yolo_head.parameters())

        # 3. Optimizer Step durchführen
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += total_step_loss

        # ====================================================================
        # --- PHASE 4: LOGGING & VISUALIZATION ---
        # ====================================================================
        if step % 50 == 0:
            writer.add_scalar('Loss/Total_Weighted', total_step_loss, global_step)
            
            for i, p_group in enumerate(optimizer.param_groups):
                grp_name = p_group.get('name', f'Group_{i}')
                writer.add_scalar(f'LR/{grp_name}', p_group['lr'], global_step)

            for k, v in step_logs.items():
                if k.startswith('weight_'): 
                    writer.add_scalar(f'Dynamic_Weights/{k}', v, global_step)
                elif k.startswith('Grad_Norm'): 
                    # 🚨 Hier werden die neuen Gradienten geloggt
                    writer.add_scalar(k, v, global_step) 
                elif k not in ['num_targets', 'max_cls', 'yolo_num_targets', 'yolo_max_cls']:
                    writer.add_scalar(f'Loss_Raw/{k}', v, global_step)
                
            writer.add_scalar('UW_Raw/LogVar_Stereo', criterion.log_var_stereo.item(), global_step)
            writer.add_scalar('UW_Raw/LogVar_Seg', criterion.log_var_seg.item(), global_step)
            writer.add_scalar('UW_Raw/LogVar_Yolo', criterion.log_var_yolo.item(), global_step)

            yolo_targets = step_logs.get('yolo_num_targets', step_logs.get('num_targets', 0))
            yolo_max_c = step_logs.get('yolo_max_cls', step_logs.get('max_cls', 0))
            
            writer.add_scalar('Debug/YOLO_Num_Targets', yolo_targets, global_step)
            writer.add_scalar('Debug/YOLO_Max_Class_ID', yolo_max_c, global_step)
            
            # IoU Histogram Plotten (falls Funktion implementiert wurde)
            if len(all_step_ious) > 0 and 'log_iou_distribution' in globals():
                log_iou_distribution(writer, torch.cat(all_step_ious), global_step)
            all_step_ious = [] # Reset nach dem Logging

        if smoothed_loss is None:
            smoothed_loss = total_step_loss
        else:
            smoothed_loss = 0.9 * smoothed_loss + 0.1 * total_step_loss

        # --- Smart Visualization Logic ---
        time_since_last = global_step - last_viz_step
        
        if best_viz_loss == float('inf'):
            is_significant_drop = True 
        else:
            is_significant_drop = (best_viz_loss - smoothed_loss) / (best_viz_loss + 1e-8) > LOSS_DROP_THRESH
        
        if global_step == 0 or (time_since_last >= MIN_VIZ_STEPS and is_significant_drop) or time_since_last >= MAX_VIZ_STEPS:
            print(f"📸 Taking snapshot at step {global_step} | Smoothed Loss: {smoothed_loss:.4f}")
            visualize_hexapod_output(step=global_step, writer=writer)
            writer.flush()
            # ✅ CHANGE 6b: VRAM freigeben nach SegFormer-Inferenz in Visualizer
            torch.cuda.empty_cache()

            model.train()
            model.backbone.eval()  # BN re-lock!
            
            best_viz_loss = min(best_viz_loss, smoothed_loss)
            last_viz_step = global_step

        pbar.set_postfix({
            'L': f"{total_step_loss:.2f}", 
            'W_Y': f"{step_logs.get('weight_yolo', 0):.2f}",
            'W_S': f"{step_logs.get('weight_stereo', 0):.2f}",
            'W_Seg': f"{step_logs.get('weight_seg', 0):.2f}"
        })

    return running_loss / steps_per_epoch_p2

# ==========================================
# --- RUN PHASE 2 (WITH CHECKPOINTING) ---
# ==========================================
import os
os.makedirs("checkpoints", exist_ok=True)
best_phase2_loss = float('inf')

print("🚀 Launching Phase 2 Refinement...")

for epoch in range(refinement_epochs):
    epoch_loss = train_phase2_one_epoch(epoch, tartan_loader, coco_loader)
    
    # Nutze deine definierte Save-Logik am Ende jeder Epoche
    save_checkpoint(
        model=model, 
        criterion=criterion,   
        optimizer=optimizer, 
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch, 
        loss=epoch_loss, 
        filename=f"checkpoint_p2_epoch_{epoch}.pth" # 🚨 Kein 'checkpoints/' davor setzen!
    )
    
    # Prüfe, ob es die bisher beste Epoche ist
    if epoch_loss < best_phase2_loss:
        best_phase2_loss = epoch_loss
        print(f"🌟 New Best Refinement Loss: {best_phase2_loss:.4f}!")
        
        # Speichere diesen Zustand als separaten "Best"-Checkpoint
        save_checkpoint(
            model=model, 
            criterion=criterion,   
            optimizer=optimizer, 
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch, 
            loss=epoch_loss, 
            filename="checkpoint_p2_best.pth" # Dieser wird immer überschrieben, wenn es einen neuen Rekord gibt
        )

print("✅ Phase 2 Complete! The Hexapod is fully sharpened.")

🚀 Launching Phase 2 Refinement...


Phase 2 - Epoch 1/12:   0%|                                                          | 0/2443 [00:00<?, ?it/s]

📸 Taking snapshot at step 0 | Smoothed Loss: 9.9553


Phase 2 - Epoch 1/12:   4%|▏     | 100/2443 [04:35<1:41:51,  2.61s/it, L=8.98, W_Y=2.00, W_S=2.00, W_Seg=0.00]

📸 Taking snapshot at step 100 | Smoothed Loss: 9.0660


...

💾 Checkpoint saved: ./checkpoints_phase2/checkpoint_p2_epoch_0.pth
🌟 New Best Refinement Loss: 8.3653!
💾 Checkpoint saved: ./checkpoints_phase2/checkpoint_p2_best.pth


Phase 2 - Epoch 2/12:  25%|█▌    | 619/2443 [28:32<1:27:51,  2.89s/it, L=8.06, W_Y=2.00, W_S=2.00, W_Seg=0.00]

📸 Taking snapshot at step 3062 | Smoothed Loss: 8.1769


Phase 2 - Epoch 2/12:  86%|████▎| 2098/2443 [1:35:12<15:39,  2.72s/it, L=8.27, W_Y=2.00, W_S=2.00, W_Seg=0.00]


KeyboardInterrupt: 

In [ ]:
%abort

In [ ]:
# Welche backbone.features-Layer liefern fl[0]-fl[3]?
print("Feature Info:")
for i, info in enumerate(model.backbone.feature_info):
    print(f"  fl[{i}]: stage={info['stage']}, reduction={info['reduction']}, "
          f"module={info['module']}")

# Welche Layer sind aktuell trainierbar?
print("\nTrainable backbone layers:")
for name, param in model.backbone.named_parameters():
    if 'blocks' in name:
        try:
            idx = int(name.split('.')[1])
            if param.requires_grad:
                print(f"  TRAINABLE: {name.split('.')[0]}.{idx}")
                break
        except: pass

# Kurz: ab welchem Index wird trainiert?
frozen, trainable = [], []
for name, param in model.backbone.named_parameters():
    if 'blocks' in name:
        try:
            idx = int(name.split('.')[1])
            (trainable if param.requires_grad else frozen).append(idx)
        except: pass
print(f"Frozen layers:   {sorted(set(frozen))}")
print(f"Trainable layers: {sorted(set(trainable))}")